# Height-map target extraction — Prior / Parameter Sensitivity Audit (Notebook 04)

Person B (height-map extraction). **Tracks 8, 10, 14 only**. **Track 21 is sealed and must not be loaded**.

Purpose: replicate Notebook 03 “CONTROL” behavior and run targeted sensitivity experiments to quantify how much selection depends on priors/configs:
- central-y prior (band + score)
- preferred-width prior (score)
- threshold multiplier (MAD × k)
- scoring-term ablations

Constraints:
- No virtual env creation.
- Do not modify organizer notebooks/files.
- Do not implement production extractor in `src/`.
- Do not create ML models.
- Do not analyze Track 21.

All numerical conclusions in this notebook are computed from executed cells and saved artifacts.

In [1]:
import sys
print(sys.executable)

c:\Program Files\Python313\python.exe


In [2]:
# Section 1: Repository/Notebook/Outputs Inspection Checklist (read-only)
from pathlib import Path
from datetime import datetime
import sys, json, math, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=RuntimeWarning)


def find_project_dir(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    here = Path(__file__).resolve() if '__file__' in globals() else None
    if here is not None:
        candidates += [here.parent, *here.parents]
    for p in candidates:
        if (p / 'src' / 'nsf_fmrg_data.py').exists() and (p / 'data' / 'raw' / 'height_maps').exists():
            return p
    raise RuntimeError('Could not resolve project root containing src/nsf_fmrg_data.py and data/raw/height_maps')


PROJECT_DIR = find_project_dir()
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = PROJECT_DIR / 'processed_data' / 'run_outputs' / f'04_heightmap_prior_sensitivity_{RUN_TAG}'
TABLE_DIR = OUT_DIR / 'tables'
FIG_DIR = OUT_DIR / 'figures'
DIAG_DIR = OUT_DIR / 'diagnostics'
for d in [TABLE_DIR, FIG_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('OUT_DIR     =', OUT_DIR)

# Explicit constraints
SEALED_TRACKS = {21}
TRACK_IDS = [8, 10, 14]
assert not (set(TRACK_IDS) & SEALED_TRACKS), 'Track 21 is sealed and must not be analyzed.'

REQUESTED_X_MM = np.array([25, 35, 45, 55, 65, 75, 85, 95], dtype=float)
print('TRACK_IDS =', TRACK_IDS)
print('REQUESTED_X_MM =', REQUESTED_X_MM.tolist())

# Run-output inspection helper
RUN_OUTPUTS_DIR = PROJECT_DIR / 'processed_data' / 'run_outputs'
print('RUN_OUTPUTS_DIR =', RUN_OUTPUTS_DIR)

runs = []
if RUN_OUTPUTS_DIR.exists():
    for p in RUN_OUTPUTS_DIR.iterdir():
        if p.is_dir():
            runs.append(p)

runs_sorted = sorted(runs, key=lambda p: p.name)
print('Found run_outputs dirs:', len(runs_sorted))
for p in runs_sorted[-10:]:
    print(' -', p.name)

# Pick the most recent notebook-03 run directory if present
nb03_runs = [p for p in runs_sorted if p.name.startswith('03_heightmap_target_extraction_exploration_')]
NB03_DIR = nb03_runs[-1] if nb03_runs else None
print('NB03_DIR =', NB03_DIR)

if NB03_DIR is not None:
    cand = NB03_DIR / 'tables' / 'run_summary.json'
    if cand.exists():
        summary = json.loads(cand.read_text())
        print('Notebook-03 run_summary.json found. Keys:', sorted(summary.keys()))
        print('Notebook-03 out_dir:', summary.get('out_dir'))
    else:
        print('WARNING: Notebook-03 run_summary.json not found under', cand)
else:
    print('WARNING: No notebook-03 run dir found under processed_data/run_outputs')

# Guardrails: do not analyze Track 21
print('SEALED_TRACKS =', SEALED_TRACKS)

PROJECT_DIR = c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge
OUT_DIR     = c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352
TRACK_IDS = [8, 10, 14]
REQUESTED_X_MM = [25.0, 35.0, 45.0, 55.0, 65.0, 75.0, 85.0, 95.0]
RUN_OUTPUTS_DIR = c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs
Found run_outputs dirs: 8
 - 03_heightmap_target_extraction_exploration_20260717_112304
 - 04_heightmap_prior_sensitivity_20260717_112352
 - 05_heightmap_persistence_corridor_20260717_110359
 - 20260713_234646
 - 20260714_000607
 - 20260714_000803
 - 20260714_001638
 - 20260714_105929
NB03_DIR = c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\03_heightmap_target_extraction_exploration_20260717_112304
Notebook-03 run_summary.json found. Keys: ['config', 'diagnostic_count', 'out_dir', 'overview_count', 's

## 2) Exact trace of Notebook-03 pipeline (component generation → scoring → ambiguity → gradient refinement)

This section recreates the relevant helper functions **verbatim in behavior** (Notebook-03 local helpers) so we can (a) reproduce CONTROL outputs and (b) vary one thing at a time.

We will keep the same function boundaries as Notebook 03:
- `baseline_threshold` (substrate baseline + MAD)
- `candidate_components` (thresholded connected components + width filters)
- `score_components` (height + width + center + continuity; rank + ambiguity)
- `refine_edges` (finite-run-preserving smoothing + gradient edge pick)

No NaN filling: we carry `valid = isfinite(z)` and only compute smoothing/gradients within finite runs.

In [3]:
# Notebook-03 helper logic (replicated)

def robust_mad(values, scale=True):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan
    med = np.nanmedian(v)
    mad = np.nanmedian(np.abs(v - med))
    return float(1.4826 * mad if scale else mad)


def finite_runs(mask):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    if idx.size == 0:
        return []
    breaks = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[breaks + 1]]
    stops = np.r_[idx[breaks] + 1, idx[-1] + 1]
    return [(int(a), int(b)) for a, b in zip(starts, stops)]


def smooth_within_finite_runs(z, valid, window, min_run):
    z = np.asarray(z, dtype=float)
    out = np.full_like(z, np.nan, dtype=float)
    w = int(window)
    if w < 3 or w % 2 == 0:
        w = max(3, w + 1)
    kernel = np.ones(w, dtype=float) / w
    for a, b in finite_runs(valid):
        run = z[a:b]
        if (b - a) >= min_run:
            pad = w // 2
            padded = np.pad(run, pad_width=pad, mode='edge')
            out[a:b] = np.convolve(padded, kernel, mode='valid')
        else:
            out[a:b] = run
    return out


def gradients_within_finite_runs(y, z_smooth, valid, min_run):
    grad = np.full_like(z_smooth, np.nan, dtype=float)
    for a, b in finite_runs(valid & np.isfinite(z_smooth)):
        if (b - a) >= min_run:
            grad[a:b] = np.gradient(z_smooth[a:b], y[a:b])
    return grad


def baseline_threshold(y, z_um, valid, config):
    sub = valid & ((y < config['central_exclusion_y_min_mm']) | (y > config['central_exclusion_y_max_mm']))
    source = 'substrate_outside_exclusion'
    if int(sub.sum()) < config['min_baseline_samples']:
        sub = valid.copy()
        source = 'all_finite_fallback'
    vals = z_um[sub]
    if vals.size == 0:
        return np.nan, np.nan, np.nan, source
    base = float(np.nanmedian(vals))
    spread = max(float(robust_mad(vals)), config['min_spread_um'])
    thr = base + config['threshold_mad_multiplier'] * spread
    return base, spread, float(thr), source


def candidate_components(y, z_um, valid, threshold, baseline, config):
    cand = valid & np.isfinite(z_um) & (z_um >= threshold)
    comps = []
    for a, b in finite_runs(cand):
        n = b - a
        ymin = float(y[a])
        ymax = float(y[b - 1])
        width = float(ymax - ymin)
        if n < config['min_component_samples'] or width < config['min_component_width_mm'] or width > config['max_component_width_mm']:
            continue
        vals = z_um[a:b]
        comps.append({
            'a': a, 'b': b, 'n': int(n), 'y_min': ymin, 'y_max': ymax, 'width_mm': width,
            'centroid_mm': float(np.nanmean(y[a:b])),
            'median_above_baseline_um': float(np.nanmedian(vals - baseline)),
            'peak_above_baseline_um': float(np.nanmax(vals - baseline)),
        })
    return cand, comps


def score_components(comps, prev_center, config):
    prior_center = 0.5 * (config['central_prior_min_mm'] + config['central_prior_max_mm'])
    prior_half = 0.5 * (config['central_prior_max_mm'] - config['central_prior_min_mm'])
    scored = []
    for c in comps:
        center_norm = min(abs(c['centroid_mm'] - prior_center) / max(prior_half, 1e-9), 2.0)
        center_score = max(0.0, 1.0 - 0.5 * center_norm)
        height_score = math.tanh(max(c['median_above_baseline_um'], 0.0) / 12.0)
        width_score = math.exp(-((c['width_mm'] - config['preferred_width_mm']) / config['width_scale_mm']) ** 2)
        if prev_center is None or not np.isfinite(prev_center):
            continuity_score = 0.5
        else:
            continuity_score = math.exp(-abs(c['centroid_mm'] - prev_center) / config['continuity_scale_mm'])
        score = (config['score_height_weight'] * height_score +
                 config['score_width_weight'] * width_score +
                 config['score_center_weight'] * center_score +
                 config['score_continuity_weight'] * continuity_score)
        cc = dict(c)
        cc.update({
            'score': float(score),
            'center_score': float(center_score),
            'height_score': float(height_score),
            'width_score': float(width_score),
            'continuity_score': float(continuity_score),
        })
        scored.append(cc)
    scored.sort(key=lambda d: d['score'], reverse=True)
    if not scored:
        return None, np.nan, scored, 'no_valid_component'
    best = scored[0]
    second = scored[1]['score'] if len(scored) > 1 else np.nan
    ambiguity = float(second / best['score']) if np.isfinite(second) and best['score'] > 0 else 0.0
    if best['score'] < config['min_selected_score']:
        return best, ambiguity, scored, 'low_score'
    if np.isfinite(ambiguity) and ambiguity >= config['ambiguity_ratio']:
        return best, ambiguity, scored, 'ambiguous'
    return best, ambiguity, scored, 'selected'


def refine_edges(y, z_um, valid, comp, config):
    z_s = smooth_within_finite_runs(z_um, valid, config['smooth_window_samples'], config['min_run_samples_for_smoothing'])
    grad = gradients_within_finite_runs(y, z_s, valid, config['min_run_samples_for_smoothing'])
    margin = config['edge_search_margin_mm']
    out = {
        'refined_left_mm': np.nan,
        'refined_right_mm': np.nan,
        'left_edge_strength': np.nan,
        'right_edge_strength': np.nan,
        'grad': grad,
        'z_smooth': z_s,
    }
    if comp is None:
        return out
    left_zone = (y >= comp['y_min'] - margin) & (y <= comp['y_min'] + margin) & np.isfinite(grad)
    right_zone = (y >= comp['y_max'] - margin) & (y <= comp['y_max'] + margin) & np.isfinite(grad)
    if left_zone.any():
        idx = np.flatnonzero(left_zone)
        j = idx[int(np.nanargmax(grad[idx]))]
        strength = float(grad[j])
        if strength >= config['edge_min_strength_um_per_mm']:
            out['refined_left_mm'] = float(y[j])
        out['left_edge_strength'] = strength
    if right_zone.any():
        idx = np.flatnonzero(right_zone)
        j = idx[int(np.nanargmin(grad[idx]))]
        strength = float(abs(grad[j]))
        if strength >= config['edge_min_strength_um_per_mm']:
            out['refined_right_mm'] = float(y[j])
        out['right_edge_strength'] = strength
    return out


def analyze_cross_section(track_id, x_req, x_sel, y, z_um, config, prev_center=None):
    valid = np.isfinite(z_um)
    finite_fraction = float(valid.mean())

    baseline, spread, threshold, baseline_source = baseline_threshold(y, z_um, valid, config)
    if np.isfinite(threshold):
        cand_mask, comps = candidate_components(y, z_um, valid, threshold, baseline, config)
    else:
        cand_mask, comps = np.zeros_like(valid), []

    best, ambiguity, scored, status = score_components(comps, prev_center, config)
    allow_boundaries = status == 'selected'

    edge = refine_edges(y, z_um, valid, best if allow_boundaries else None, config)

    row = {
        'track_id': int(track_id),
        'requested_x_mm': float(x_req),
        'selected_x_mm': float(x_sel),
        'finite_fraction': finite_fraction,
        'baseline_um': baseline,
        'robust_spread_um': spread,
        'threshold_um': threshold,
        'baseline_source': baseline_source,
        'fraction_finite_above_threshold': float(cand_mask.sum() / max(valid.sum(), 1)),
        'n_candidate_components': int(len(scored)),
        'selected_component_score': np.nan if best is None else float(best['score']),
        'ambiguity_score': float(ambiguity) if np.isfinite(ambiguity) else np.nan,
        'segmentation_left_mm': np.nan,
        'segmentation_right_mm': np.nan,
        'segmentation_width_mm': np.nan,
        'segmentation_centroid_mm': np.nan,
        'refined_left_mm': np.nan,
        'refined_right_mm': np.nan,
        'refined_width_mm': np.nan,
        'left_edge_strength': edge['left_edge_strength'],
        'right_edge_strength': edge['right_edge_strength'],
        'extraction_status': status,
    }

    if allow_boundaries and best is not None:
        row['segmentation_left_mm'] = float(best['y_min'])
        row['segmentation_right_mm'] = float(best['y_max'])
        row['segmentation_width_mm'] = float(best['width_mm'])
        row['segmentation_centroid_mm'] = float(best['centroid_mm'])
        row['refined_left_mm'] = float(edge['refined_left_mm'])
        row['refined_right_mm'] = float(edge['refined_right_mm'])
        if np.isfinite(edge['refined_left_mm']) and np.isfinite(edge['refined_right_mm']) and edge['refined_right_mm'] > edge['refined_left_mm']:
            row['refined_width_mm'] = float(edge['refined_right_mm'] - edge['refined_left_mm'])

    detail = {
        'valid': valid,
        'candidate_mask': cand_mask,
        'components': scored,
        'selected': best if allow_boundaries else None,
        'edge': edge,
    }
    return row, detail

print('Replicated Notebook-03 pipeline helpers loaded.')

Replicated Notebook-03 pipeline helpers loaded.


## 3) Parameter inventory + prior-encoding parameters

We explicitly list every Notebook-03 parameter that influences:
- component generation
- scoring / ranking
- ambiguity / rejection
- gradient refinement

We also label which ones encode *geometric priors* (expected y location, expected width, centrality).

In [4]:
# CONTROL config (must match Notebook 03)
CONTROL_CONFIG = {
    'central_exclusion_y_min_mm': 0.65,
    'central_exclusion_y_max_mm': 1.35,
    'central_prior_min_mm': 0.65,
    'central_prior_max_mm': 1.35,
    'threshold_mad_multiplier': 2.5,
    'min_spread_um': 0.75,
    'min_baseline_samples': 30,
    'smooth_window_samples': 9,
    'min_run_samples_for_smoothing': 15,
    'min_component_samples': 6,
    'min_component_width_mm': 0.035,
    'max_component_width_mm': 1.10,
    'score_height_weight': 2.0,
    'score_width_weight': 1.0,
    'score_center_weight': 1.5,
    'score_continuity_weight': 0.6,
    'preferred_width_mm': 0.32,
    'width_scale_mm': 0.30,
    'continuity_scale_mm': 0.18,
    'min_selected_score': 1.0,
    'ambiguity_ratio': 0.85,
    'edge_search_margin_mm': 0.08,
    'edge_min_strength_um_per_mm': 50.0,
}

PARAM_METADATA = [
    # generation / detrend baseline selection
    ('central_exclusion_y_min_mm', 'baseline_threshold + substrate-focused detrend plane-fit mask', 'generation', 'prior: expected y (excluded central band)'),
    ('central_exclusion_y_max_mm', 'baseline_threshold + substrate-focused detrend plane-fit mask', 'generation', 'prior: expected y (excluded central band)'),
    ('min_baseline_samples', 'baseline_threshold', 'generation', 'none'),
    ('min_spread_um', 'baseline_threshold', 'generation', 'none'),
    ('threshold_mad_multiplier', 'baseline_threshold', 'generation', 'none'),

    ('min_component_samples', 'candidate_components', 'generation', 'none'),
    ('min_component_width_mm', 'candidate_components', 'generation', 'prior: expected width (hard bounds)'),
    ('max_component_width_mm', 'candidate_components', 'generation', 'prior: expected width (hard bounds)'),

    # scoring
    ('score_height_weight', 'score_components', 'scoring', 'none'),
    ('score_width_weight', 'score_components', 'scoring', 'prior: expected width (soft preference)'),
    ('preferred_width_mm', 'score_components (width_score)', 'scoring', 'prior: expected width (soft preference center)'),
    ('width_scale_mm', 'score_components (width_score)', 'scoring', 'prior strength for width (soft)'),

    ('score_center_weight', 'score_components', 'scoring', 'prior: centrality/y location'),
    ('central_prior_min_mm', 'score_components (center_score)', 'scoring', 'prior: expected y band'),
    ('central_prior_max_mm', 'score_components (center_score)', 'scoring', 'prior: expected y band'),

    ('score_continuity_weight', 'score_components', 'scoring', 'weak prior: temporal/x continuity'),
    ('continuity_scale_mm', 'score_components (continuity_score)', 'scoring', 'continuity strength (not geometric prior)'),

    # ambiguity / rejection
    ('min_selected_score', 'score_components', 'rejection', 'none'),
    ('ambiguity_ratio', 'score_components', 'rejection', 'none'),

    # gradient refinement
    ('smooth_window_samples', 'refine_edges', 'refinement', 'none'),
    ('min_run_samples_for_smoothing', 'refine_edges + gradients_within_finite_runs', 'refinement', 'none'),
    ('edge_search_margin_mm', 'refine_edges', 'refinement', 'prior-ish: search locality near segmentation edge'),
    ('edge_min_strength_um_per_mm', 'refine_edges', 'refinement', 'none'),
]

param_df = pd.DataFrame(PARAM_METADATA, columns=['parameter', 'where_used', 'effect_category', 'prior_encoding'])
param_df['control_value'] = param_df['parameter'].map(CONTROL_CONFIG)
param_df = param_df[['parameter', 'control_value', 'where_used', 'effect_category', 'prior_encoding']]

display(param_df)
param_df.to_csv(TABLE_DIR / 'parameter_inventory.csv', index=False)
print('Saved:', TABLE_DIR / 'parameter_inventory.csv')

# Precise explanation of how center/width priors influence selection
prior_center = 0.5 * (CONTROL_CONFIG['central_prior_min_mm'] + CONTROL_CONFIG['central_prior_max_mm'])
prior_half = 0.5 * (CONTROL_CONFIG['central_prior_max_mm'] - CONTROL_CONFIG['central_prior_min_mm'])
print('CONTROL center prior center=', prior_center, 'halfwidth=', prior_half)
print('CONTROL preferred_width_mm =', CONTROL_CONFIG['preferred_width_mm'])

explain = pd.DataFrame([
    {
        'term': 'center_score',
        'formula': 'max(0, 1 - 0.5 * min(|centroid - prior_center|/prior_half, 2))',
        'effect': 'Linear drop from 1 at prior_center to 0.5 at band edge; 0 at 2× band halfwidth away (clipped).',
        'prior_params': 'central_prior_min_mm, central_prior_max_mm, score_center_weight',
    },
    {
        'term': 'width_score',
        'formula': 'exp(-((width - preferred_width)/width_scale)^2)',
        'effect': 'Gaussian-like preference peaked at preferred_width; decays with width_scale.',
        'prior_params': 'preferred_width_mm, width_scale_mm, score_width_weight',
    },
])

display(explain)
explain.to_csv(TABLE_DIR / 'prior_term_formulas.csv', index=False)
print('Saved:', TABLE_DIR / 'prior_term_formulas.csv')

,parameter,control_value,where_used,effect_category,prior_encoding
0,central_exclusion_y_min_mm,0.650,baseline_threshold + substrate-focused detrend...,generation,prior: expected y (excluded central band)
1,central_exclusion_y_max_mm,1.350,baseline_threshold + substrate-focused detrend...,generation,prior: expected y (excluded central band)
2,min_baseline_samples,30.000,baseline_threshold,generation,none
3,min_spread_um,0.750,baseline_threshold,generation,none
4,threshold_mad_multiplier,2.500,baseline_threshold,generation,none
5,min_component_samples,6.000,candidate_components,generation,none
6,min_component_width_mm,0.035,candidate_components,generation,prior: expected width (hard bounds)
7,max_component_width_mm,1.100,candidate_components,generation,prior: expected width (hard bounds)
8,score_height_weight,2.000,score_components,scoring,none
9,score_width_weight,1.000,score_components,scoring,prior: expected width (soft preference)


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\parameter_inventory.csv
CONTROL center prior center= 1.0 halfwidth= 0.35000000000000003
CONTROL preferred_width_mm = 0.32


,term,formula,effect,prior_params
0,center_score,"max(0, 1 - 0.5 * min(|centroid - prior_center|...",Linear drop from 1 at prior_center to 0.5 at b...,"central_prior_min_mm, central_prior_max_mm, sc..."
1,width_score,exp(-((width - preferred_width)/width_scale)^2),Gaussian-like preference peaked at preferred_w...,"preferred_width_mm, width_scale_mm, score_widt..."


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\prior_term_formulas.csv


## 4) Data loading + case definition (Tracks 8/10/14 only; fixed x positions)

We load height maps with the existing loader (`src/nsf_fmrg_data.py`). We define the 24 cases as:

$\{8,10,14\} \times \{25,35,45,55,65,75,85,95\}\,\text{mm}$

No NaN filling: finite masks are explicit throughout.

In [5]:
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsf_fmrg_data import load_wyko_asc

HEIGHT_DIR = PROJECT_DIR / 'data' / 'raw' / 'height_maps'

# Case list (track, requested_x)
CASES = [(int(t), float(x)) for t in TRACK_IDS for x in REQUESTED_X_MM]
case_df = pd.DataFrame(CASES, columns=['track_id', 'requested_x_mm'])
display(case_df)

# Load raw height maps (once)
height_data = {}
for track_id in TRACK_IDS:
    assert track_id not in SEALED_TRACKS
    h = load_wyko_asc(HEIGHT_DIR, track_id, crop_to_common=True)
    x = np.asarray(h['x_actual_mm'], dtype=float)
    y = np.asarray(h['y_mm'], dtype=float)
    Z = np.asarray(h['Z_mm'], dtype=float)
    height_data[int(track_id)] = {'raw': h, 'x': x, 'y': y, 'Z_mm': Z}

print('Loaded tracks:', sorted(height_data.keys()))
print('Example shapes:', {t: height_data[t]['Z_mm'].shape for t in height_data})

# Nearest-column selection per requested x
case_to_col = {}
for track_id, x_req in CASES:
    x = height_data[track_id]['x']
    j = int(np.argmin(np.abs(x - x_req)))
    case_to_col[(track_id, x_req)] = j

# Safety: never allow Track 21 in data dict
assert 21 not in height_data

,track_id,requested_x_mm
0,8,25.0
1,8,35.0
2,8,45.0
3,8,55.0
4,8,65.0
5,8,75.0
6,8,85.0
7,8,95.0
8,10,25.0
9,10,35.0


Loaded tracks: [8, 10, 14]
Example shapes: {8: (480, 20025), 10: (480, 20091), 14: (480, 19724)}


## 5) Detrending: substrate-focused detrending (working experimental method)

We match Notebook 03’s substrate-focused plane detrend:
- fit a plane using downsampled points **excluding** the central y band [0.65, 1.35] mm
- robustify the fit with iterative residual trimming

This is an exploratory detrend used only in notebooks (not moved into `src/`).

In [6]:
def fit_substrate_focused_plane(Z_mm, x_mm, y_mm, config, stride_x=40, stride_y=2):
    ylo = config['central_exclusion_y_min_mm']
    yhi = config['central_exclusion_y_max_mm']
    Zs = Z_mm[::stride_y, ::stride_x]
    xs = x_mm[::stride_x]
    ys = y_mm[::stride_y]
    Xs, Ys = np.meshgrid(xs, ys)
    z = Zs.ravel()
    A = np.c_[Xs.ravel(), Ys.ravel(), np.ones(Xs.size)]
    substrate = (Ys.ravel() < ylo) | (Ys.ravel() > yhi)
    valid = np.isfinite(z) & substrate
    n_initial = int(valid.sum())
    if n_initial < 100:
        return Z_mm.copy(), None, {'n_initial': n_initial, 'n_final': 0, 'excluded_y_band': [ylo, yhi]}
    keep = valid.copy()
    coef = None
    for _ in range(4):
        coef, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
        resid = z - A @ coef
        rv = resid[keep]
        med = np.nanmedian(rv)
        spread = max(robust_mad(rv), 1e-9)
        keep_new = valid & (np.abs(resid - med) <= 3.0 * spread)
        if keep_new.sum() < 100:
            break
        keep = keep_new
    plane = coef[0] * x_mm[None, :] + coef[1] * y_mm[:, None] + coef[2]
    return Z_mm - plane, coef, {'n_initial': n_initial, 'n_final': int(keep.sum()), 'excluded_y_band': [ylo, yhi]}


def detrend_tracks_substrate_focused(height_data, config):
    out = {}
    for track_id, bundle in height_data.items():
        x, y, Z = bundle['x'], bundle['y'], bundle['Z_mm']
        Zd, coef, meta = fit_substrate_focused_plane(Z, x, y, config)
        out[track_id] = {
            **bundle,
            'Z_detrended_mm': Zd,
            'detrend_coef': coef,
            'detrend_meta': meta,
        }
    return out


height_data_det = detrend_tracks_substrate_focused(height_data, CONTROL_CONFIG)
print('Detrended tracks (substrate-focused). Example meta:', {t: height_data_det[t]['detrend_meta'] for t in height_data_det})

# quick sanity plot for one track & one x
track_id, x_req = 8, 35.0
j = case_to_col[(track_id, x_req)]
x = height_data_det[track_id]['x']
y = height_data_det[track_id]['y']
Z_um = height_data_det[track_id]['Z_detrended_mm'][:, j] * 1000.0
valid = np.isfinite(Z_um)

fig, ax = plt.subplots(1, 1, figsize=(7.5, 3.0))
ax.plot(y[valid], Z_um[valid], '.', ms=3)
ax.axvspan(CONTROL_CONFIG['central_exclusion_y_min_mm'], CONTROL_CONFIG['central_exclusion_y_max_mm'], color='gold', alpha=0.15, lw=0, label='excluded band')
ax.set_title(f'Detrended cross-section sanity: Track {track_id}, x≈{x[ j ]:.3f} mm')
ax.set_xlabel('y (mm)')
ax.set_ylabel('detrended z (µm)')
ax.grid(alpha=0.2)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(FIG_DIR / 'sanity_detrended_example.png', dpi=160)
plt.close(fig)
print('Saved:', FIG_DIR / 'sanity_detrended_example.png')

Detrended tracks (substrate-focused). Example meta: {8: {'n_initial': 34038, 'n_final': 28823, 'excluded_y_band': [0.65, 1.35]}, 10: {'n_initial': 22414, 'n_final': 20062, 'excluded_y_band': [0.65, 1.35]}, 14: {'n_initial': 20287, 'n_final': 20287, 'excluded_y_band': [0.65, 1.35]}}
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\sanity_detrended_example.png


## 6) CONTROL configuration: reproduce Notebook-03 behavior

We run all 24 cases with CONTROL settings and store:
- full scored candidate list
- selected component (if selected)
- status (`selected`, `ambiguous`, `low_score`, `no_valid_component`)
- segmentation boundaries + optional refined edges

We additionally maintain `prev_center` per track during the 8 x-positions sweep (this matches Notebook 03’s continuity term behavior).

In [7]:
def run_extraction_for_cases(height_data_det, config, cases, case_to_col, config_name):
    rows = []
    details = {}  # (track,x_req) -> detail dict

    for track_id in sorted({t for t, _ in cases}):
        prev_center = None
        bundle = height_data_det[track_id]
        x = bundle['x']
        y = bundle['y']
        Zd_um = bundle['Z_detrended_mm'] * 1000.0

        # per-track sweep in requested x order to match continuity behavior
        for x_req in sorted([xr for (t, xr) in cases if t == track_id]):
            j = case_to_col[(track_id, x_req)]
            x_sel = float(x[j])
            z_um = Zd_um[:, j]

            row, det = analyze_cross_section(track_id, x_req, x_sel, y, z_um, config, prev_center=prev_center)
            row['config_name'] = str(config_name)
            rows.append(row)
            details[(track_id, float(x_req))] = det

            if det['selected'] is not None:
                prev_center = float(det['selected']['centroid_mm'])

    df = pd.DataFrame(rows)
    return df, details


control_df, control_details = run_extraction_for_cases(height_data_det, CONTROL_CONFIG, CASES, case_to_col, config_name='CONTROL')

display(control_df.head(10))
status_counts = control_df.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0)
display(status_counts)

control_df.to_csv(TABLE_DIR / 'control_outputs.csv', index=False)
print('Saved:', TABLE_DIR / 'control_outputs.csv')

# Save minimal details for auditing (selected component + top candidate scores)
control_comp_rows = []
for (track_id, x_req), det in control_details.items():
    sel = det['selected']
    comps = det['components']
    control_comp_rows.append({
        'track_id': track_id,
        'requested_x_mm': x_req,
        'status': 'selected' if sel is not None else control_df.query('track_id==@track_id and requested_x_mm==@x_req')['extraction_status'].iloc[0],
        'selected_y_min': np.nan if sel is None else sel['y_min'],
        'selected_y_max': np.nan if sel is None else sel['y_max'],
        'selected_centroid': np.nan if sel is None else sel['centroid_mm'],
        'selected_width': np.nan if sel is None else sel['width_mm'],
        'n_components': len(comps),
        'top_score': np.nan if len(comps) == 0 else comps[0]['score'],
        'second_score': np.nan if len(comps) < 2 else comps[1]['score'],
    })

control_comp_df = pd.DataFrame(control_comp_rows)
control_comp_df.to_csv(TABLE_DIR / 'control_selected_components.csv', index=False)
display(control_comp_df)
print('Saved:', TABLE_DIR / 'control_selected_components.csv')

,track_id,requested_x_mm,selected_x_mm,finite_fraction,baseline_um,robust_spread_um,threshold_um,baseline_source,fraction_finite_above_threshold,n_candidate_components,...,segmentation_right_mm,segmentation_width_mm,segmentation_centroid_mm,refined_left_mm,refined_right_mm,refined_width_mm,left_edge_strength,right_edge_strength,extraction_status,config_name
0,8,25.0,24.999030,0.445833,126.471900,2.883502,133.680655,substrate_outside_exclusion,0.000000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_valid_component,CONTROL
1,8,35.0,35.001814,0.627083,47.719498,0.961151,50.122374,substrate_outside_exclusion,0.475083,1,...,1.322024,0.525624,1.059212,0.736670,1.333970,0.597300,55.355642,74.599901,selected,CONTROL
2,8,45.0,45.000616,0.604167,-8.641765,1.336197,-5.301271,substrate_outside_exclusion,0.513793,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ambiguous,CONTROL
3,8,55.0,54.999418,0.764583,-32.432988,2.523906,-26.123222,substrate_outside_exclusion,0.005450,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_valid_component,CONTROL
4,8,65.0,64.998220,0.691667,-31.390476,2.084576,-26.179037,substrate_outside_exclusion,0.370482,1,...,1.365826,0.461912,1.134870,NaN,1.373790,NaN,36.876693,59.703105,selected,CONTROL
5,8,75.0,75.001004,0.700000,-22.015614,1.789526,-17.541799,substrate_outside_exclusion,0.380952,1,...,1.333970,0.489786,1.089077,0.788436,1.361844,0.573408,51.408423,59.259039,selected,CONTROL
6,8,85.0,84.999806,0.622917,5.863508,2.963222,13.271562,substrate_outside_exclusion,0.277592,1,...,1.246366,0.310596,1.091068,NaN,1.322024,NaN,30.359358,50.613680,selected,CONTROL
7,8,95.0,94.998608,0.668750,58.896844,3.146220,66.762394,substrate_outside_exclusion,0.000000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_valid_component,CONTROL
8,10,25.0,24.999030,0.000000,NaN,NaN,NaN,all_finite_fallback,0.000000,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_valid_component,CONTROL
9,10,35.0,35.001814,0.425000,55.970519,1.133125,58.803332,substrate_outside_exclusion,0.568627,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ambiguous,CONTROL


extraction_status,ambiguous,no_valid_component,selected
track_id,,,
8,1,3,4
10,3,3,2
14,0,4,4


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\control_outputs.csv


,track_id,requested_x_mm,status,selected_y_min,selected_y_max,selected_centroid,selected_width,n_components,top_score,second_score
0,8,25.0,no_valid_component,NaN,NaN,NaN,NaN,0,NaN,NaN
1,8,35.0,selected,0.796400,1.322024,1.059212,0.525624,1,3.007661,NaN
2,8,45.0,ambiguous,NaN,NaN,NaN,NaN,5,3.800324,3.555609
3,8,55.0,no_valid_component,NaN,NaN,NaN,NaN,0,NaN,NaN
4,8,65.0,selected,0.903914,1.365826,1.134870,0.461912,1,3.486367,NaN
5,8,75.0,selected,0.844184,1.333970,1.089077,0.489786,1,3.579401,NaN
6,8,85.0,selected,0.935770,1.246366,1.091068,0.310596,1,4.079073,NaN
7,8,95.0,no_valid_component,NaN,NaN,NaN,NaN,0,NaN,NaN
8,10,25.0,no_valid_component,NaN,NaN,NaN,NaN,0,NaN,NaN
9,10,35.0,ambiguous,NaN,NaN,NaN,NaN,2,3.545315,3.533327


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\control_selected_components.csv


## 7) CONTROL validation vs Notebook-03 outputs (diff + diagnosis)

We compare this notebook’s CONTROL results to the most recent Notebook-03 run artifacts (tables). We require close agreement on:
- selection/ambiguous/no-valid counts by track
- for selected cases: segmentation boundaries (within tolerance)

If mismatch is large, we stop and diagnose the cause (unit conversion, detrending differences, continuity ordering, etc.).

In [8]:
def load_nb03_representative_table(nb03_dir: Path):
    if nb03_dir is None:
        return None
    path = nb03_dir / 'tables' / 'representative_x_method_comparison.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return df


nb03_rep = load_nb03_representative_table(NB03_DIR)
if nb03_rep is None:
    print('Notebook-03 representative table not found; skipping strict validation.')
else:
    # substrate_focused rows only
    nb03_sub = nb03_rep.query("detrend_method == 'substrate_focused'").copy()

    # selection count comparison
    nb03_counts = nb03_sub.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0).sort_index()
    nb04_counts = control_df.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0).sort_index()

    print('Notebook-03 counts (substrate_focused):')
    display(nb03_counts)
    print('Notebook-04 CONTROL counts:')
    display(nb04_counts)

    counts_diff = (nb04_counts.reindex_like(nb03_counts).fillna(0) - nb03_counts).fillna(0)
    print('Counts diff (nb04 - nb03):')
    display(counts_diff)

    # boundary tolerance check for selected rows present in both
    merge = pd.merge(
        nb03_sub,
        control_df,
        on=['track_id', 'requested_x_mm'],
        how='inner',
        suffixes=('_nb03', '_nb04'),
    )

    sel_both = merge.query("extraction_status_nb03 == 'selected' and extraction_status_nb04 == 'selected'").copy()
    if len(sel_both) == 0:
        print('No cases selected in both nb03 and nb04 CONTROL.')
    else:
        for col in ['segmentation_left_mm', 'segmentation_right_mm', 'segmentation_width_mm']:
            sel_both[f'delta_{col}'] = sel_both[f'{col}_nb04'] - sel_both[f'{col}_nb03']

        display(sel_both[['track_id','requested_x_mm','selected_x_mm_nb03','selected_x_mm_nb04',
                          'delta_segmentation_left_mm','delta_segmentation_right_mm','delta_segmentation_width_mm']])

        tol = 1e-6
        max_abs = sel_both[[
            'delta_segmentation_left_mm','delta_segmentation_right_mm','delta_segmentation_width_mm'
        ]].abs().max().max()
        print('Max abs boundary delta:', max_abs)

        # For reproducibility we expect exact matches (same code), but allow tiny float diffs.
        if max_abs > 1e-4:
            print('WARNING: CONTROL reproduction differs noticeably from notebook 03. Diagnose before proceeding.')
        else:
            print('CONTROL reproduction matches notebook 03 within tolerance.')

    # Save validation merge
    if nb03_rep is not None:
        merge.to_csv(TABLE_DIR / 'control_vs_nb03_merge.csv', index=False)
        print('Saved:', TABLE_DIR / 'control_vs_nb03_merge.csv')

Notebook-03 counts (substrate_focused):


extraction_status,ambiguous,no_valid_component,selected
track_id,,,
8,1,3,4
10,3,3,2
14,0,4,4


Notebook-04 CONTROL counts:


extraction_status,ambiguous,no_valid_component,selected
track_id,,,
8,1,3,4
10,3,3,2
14,0,4,4


Counts diff (nb04 - nb03):


extraction_status,ambiguous,no_valid_component,selected
track_id,,,
8,0,0,0
10,0,0,0
14,0,0,0


,track_id,requested_x_mm,selected_x_mm_nb03,selected_x_mm_nb04,delta_segmentation_left_mm,delta_segmentation_right_mm,delta_segmentation_width_mm
1,8,35.0,35.001814,35.001814,0.000000e+00,0.0,0.000000e+00
4,8,65.0,64.998220,64.998220,1.110223e-16,0.0,0.000000e+00
5,8,75.0,75.001004,75.001004,0.000000e+00,0.0,5.551115e-17
6,8,85.0,84.999806,84.999806,1.110223e-16,0.0,0.000000e+00
11,10,55.0,54.999418,54.999418,0.000000e+00,0.0,0.000000e+00
13,10,75.0,75.001004,75.001004,1.110223e-16,0.0,5.551115e-17
17,14,35.0,35.001814,35.001814,0.000000e+00,0.0,5.551115e-17
19,14,55.0,54.999418,54.999418,0.000000e+00,0.0,0.000000e+00
20,14,65.0,64.998220,64.998220,0.000000e+00,0.0,5.551115e-17
22,14,85.0,84.999806,84.999806,0.000000e+00,0.0,0.000000e+00


Max abs boundary delta: 1.1102230246251565e-16
CONTROL reproduction matches notebook 03 within tolerance.
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\control_vs_nb03_merge.csv


## 8) Central-y prior sensitivity experiments (A–E)

We vary only center-prior related settings/logic:
- A CONTROL
- B NO CENTER SCORE (center weight=0)
- C WEAK CENTER (center weight=0.25)
- D BROAD CENTER PRIOR (broaden the band using pooled-data heuristic; single band for all tracks)
- E DATA-DERIVED CENTER (derive plausible band from pooled candidate centroid distribution under weak center)

Data-derived approach is documented and does **not** use per-track labels or manual boundaries.

In [9]:
def config_with_overrides(base, **overrides):
    c = dict(base)
    c.update(overrides)
    return c


def pooled_candidate_centroids(height_data_det, config, cases, case_to_col):
    # Collect candidate centroids across all cases WITHOUT using final selection.
    cents = []
    for (track_id, x_req) in cases:
        bundle = height_data_det[track_id]
        x = bundle['x']
        y = bundle['y']
        Zd_um = bundle['Z_detrended_mm'] * 1000.0
        j = case_to_col[(track_id, x_req)]
        z_um = Zd_um[:, j]
        valid = np.isfinite(z_um)
        baseline, spread, thr, _ = baseline_threshold(y, z_um, valid, config)
        if not np.isfinite(thr):
            continue
        _, comps = candidate_components(y, z_um, valid, thr, baseline, config)
        for c in comps:
            if np.isfinite(c['centroid_mm']):
                cents.append(float(c['centroid_mm']))
    return np.array(cents, dtype=float)


def derive_center_band_from_centroids(centroids, k_mad=3.5, min_width=0.6):
    # Transparent, pooled-data-derived band: median +/- k*MAD, with a minimum band width.
    c = np.asarray(centroids, dtype=float)
    c = c[np.isfinite(c)]
    if c.size == 0:
        return (0.65, 1.35, {'method': 'empty_fallback', 'n': 0})
    med = float(np.nanmedian(c))
    mad = float(robust_mad(c))
    half = max(k_mad * mad, 0.5 * min_width)
    lo, hi = med - half, med + half
    meta = {'method': 'median_mad', 'n': int(c.size), 'median': med, 'mad': mad, 'k_mad': float(k_mad), 'min_width': float(min_width)}
    return (float(lo), float(hi), meta)


def summarize_against_control(df, control_df):
    key = ['track_id', 'requested_x_mm']
    m = pd.merge(control_df[key + ['extraction_status', 'segmentation_left_mm', 'segmentation_right_mm']],
                 df[key + ['extraction_status', 'segmentation_left_mm', 'segmentation_right_mm']],
                 on=key, how='inner', suffixes=('_control', '_run'))

    def same_component(row, tol=1e-6):
        if row['extraction_status_control'] != 'selected' or row['extraction_status_run'] != 'selected':
            return False
        return (abs(row['segmentation_left_mm_control'] - row['segmentation_left_mm_run']) <= tol and
                abs(row['segmentation_right_mm_control'] - row['segmentation_right_mm_run']) <= tol)

    m['same_as_control'] = m.apply(same_component, axis=1)
    m['status_changed'] = m['extraction_status_run'] != m['extraction_status_control']
    m['selected_changed'] = (m['extraction_status_run'].eq('selected') & m['extraction_status_control'].eq('selected') & ~m['same_as_control'])
    return m


# Build center-prior configs
center_configs = {}
center_configs['A_CONTROL'] = dict(CONTROL_CONFIG)
center_configs['B_NO_CENTER_SCORE'] = config_with_overrides(CONTROL_CONFIG, score_center_weight=0.0)
center_configs['C_WEAK_CENTER'] = config_with_overrides(CONTROL_CONFIG, score_center_weight=0.25)

# D broad center prior: use pooled centroid distribution under CONTROL to make band broader
centroids_control = pooled_candidate_centroids(height_data_det, CONTROL_CONFIG, CASES, case_to_col)
lo_broad, hi_broad, meta_broad = derive_center_band_from_centroids(centroids_control, k_mad=6.0, min_width=1.2)
center_configs['D_BROAD_CENTER_PRIOR'] = config_with_overrides(CONTROL_CONFIG, central_prior_min_mm=lo_broad, central_prior_max_mm=hi_broad)

# E data-derived center: compute centroids under weak center (less prior pull) then define band
centroids_weak = pooled_candidate_centroids(height_data_det, center_configs['C_WEAK_CENTER'], CASES, case_to_col)
lo_dd, hi_dd, meta_dd = derive_center_band_from_centroids(centroids_weak, k_mad=3.5, min_width=0.8)
center_configs['E_DATA_DERIVED_CENTER'] = config_with_overrides(CONTROL_CONFIG, central_prior_min_mm=lo_dd, central_prior_max_mm=hi_dd)

center_meta = {
    'D_BROAD_CENTER_PRIOR': meta_broad,
    'E_DATA_DERIVED_CENTER': meta_dd,
}
(Path(TABLE_DIR) / 'center_prior_derivation.json').write_text(json.dumps(center_meta, indent=2))
print('Saved:', TABLE_DIR / 'center_prior_derivation.json')
print('Derived bands:')
print(' D broad band:', (lo_broad, hi_broad), meta_broad)
print(' E data-derived band:', (lo_dd, hi_dd), meta_dd)

# Run all center configs
center_runs = {}
center_merges = {}
for name, cfg in center_configs.items():
    df, det = run_extraction_for_cases(height_data_det, cfg, CASES, case_to_col, config_name=name)
    center_runs[name] = {'df': df, 'details': det, 'config': cfg}
    center_merges[name] = summarize_against_control(df, control_df)

# Summaries
rows = []
for name, run in center_runs.items():
    df = run['df']
    counts = df.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0)
    for track_id in TRACK_IDS:
        r = {'config_name': name, 'track_id': track_id}
        for status in ['selected', 'ambiguous', 'no_valid_component', 'low_score']:
            r[status] = int(counts.loc[track_id, status]) if (track_id in counts.index and status in counts.columns) else 0
        rows.append(r)

center_summary = pd.DataFrame(rows)
display(center_summary)
center_summary.to_csv(TABLE_DIR / 'center_prior_summary_counts.csv', index=False)
print('Saved:', TABLE_DIR / 'center_prior_summary_counts.csv')

# Distributions
def selected_distributions(df):
    sel = df.query("extraction_status == 'selected'").copy()
    return sel[['segmentation_centroid_mm', 'segmentation_width_mm']].dropna()

center_dists = []
for name, run in center_runs.items():
    dist = selected_distributions(run['df'])
    center_dists.append({
        'config_name': name,
        'n_selected': int(len(dist)),
        'centroid_median': float(np.nanmedian(dist['segmentation_centroid_mm'])) if len(dist) else np.nan,
        'centroid_mad': float(robust_mad(dist['segmentation_centroid_mm'])) if len(dist) else np.nan,
        'width_median': float(np.nanmedian(dist['segmentation_width_mm'])) if len(dist) else np.nan,
        'width_mad': float(robust_mad(dist['segmentation_width_mm'])) if len(dist) else np.nan,
    })
center_dists_df = pd.DataFrame(center_dists)
display(center_dists_df)
center_dists_df.to_csv(TABLE_DIR / 'center_prior_selected_distributions.csv', index=False)

# How often selection differs from CONTROL
diff_rows = []
for name, m in center_merges.items():
    diff_rows.append({
        'config_name': name,
        'n_cases': int(len(m)),
        'n_status_changed': int(m['status_changed'].sum()),
        'n_selected_changed': int(m['selected_changed'].sum()),
        'n_same_as_control': int(m['same_as_control'].sum()),
    })
center_diff_df = pd.DataFrame(diff_rows)
display(center_diff_df)
center_diff_df.to_csv(TABLE_DIR / 'center_prior_diff_vs_control.csv', index=False)
print('Saved:', TABLE_DIR / 'center_prior_diff_vs_control.csv')

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\center_prior_derivation.json
Derived bands:
 D broad band: (0.36847755560000217, 1.7499464443999975) {'method': 'median_mad', 'n': 25, 'median': 1.0592119999999998, 'mad': 0.11512240739999961, 'k_mad': 6.0, 'min_width': 1.2}
 E data-derived band: (0.6562835741000012, 1.4621404258999986) {'method': 'median_mad', 'n': 25, 'median': 1.0592119999999998, 'mad': 0.11512240739999961, 'k_mad': 3.5, 'min_width': 0.8}


,config_name,track_id,selected,ambiguous,no_valid_component,low_score
0,A_CONTROL,8,4,1,3,0
1,A_CONTROL,10,2,3,3,0
2,A_CONTROL,14,4,0,4,0
3,B_NO_CENTER_SCORE,8,4,1,3,0
4,B_NO_CENTER_SCORE,10,3,2,3,0
5,B_NO_CENTER_SCORE,14,4,0,4,0
6,C_WEAK_CENTER,8,4,1,3,0
7,C_WEAK_CENTER,10,3,2,3,0
8,C_WEAK_CENTER,14,4,0,4,0
9,D_BROAD_CENTER_PRIOR,8,4,1,3,0


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\center_prior_summary_counts.csv


,config_name,n_selected,centroid_median,centroid_mad,width_median,width_mad
0,A_CONTROL,10,1.039302,0.075272,0.316569,0.180063
1,B_NO_CENTER_SCORE,11,1.027356,0.091508,0.310596,0.153497
2,C_WEAK_CENTER,11,1.027356,0.091508,0.310596,0.153497
3,D_BROAD_CENTER_PRIOR,9,1.051248,0.059037,0.322542,0.135785
4,E_DATA_DERIVED_CENTER,9,1.051248,0.059037,0.322542,0.135785


,config_name,n_cases,n_status_changed,n_selected_changed,n_same_as_control
0,A_CONTROL,24,0,0,10
1,B_NO_CENTER_SCORE,24,1,0,10
2,C_WEAK_CENTER,24,1,0,10
3,D_BROAD_CENTER_PRIOR,24,1,0,9
4,E_DATA_DERIVED_CENTER,24,1,0,9


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\center_prior_diff_vs_control.csv


## 9) Preferred-width prior sensitivity experiments (A–E + perturbed sweep)

We vary only width-prior related settings/logic while holding other CONTROL parameters fixed:
- A CONTROL
- B NO WIDTH SCORE (width weight=0)
- C BROAD WIDTH PREFERENCE (replace sharp Gaussian with broad, slowly varying score)
- D DATA-DERIVED WIDTH PRIOR (derive pooled width tendency from candidate widths)
- E PERTURBED preferred widths sweep: 0.20, 0.26, 0.32, 0.40, 0.50 mm

Implementation note: for C and D we keep candidate generation unchanged; only the *width score term* is changed.

In [10]:
# To alter width scoring without rewriting the whole pipeline, we provide a variant scorer.

def score_components_with_custom_width(comps, prev_center, config, width_mode='gaussian', width_params=None):
    width_params = {} if width_params is None else dict(width_params)

    prior_center = 0.5 * (config['central_prior_min_mm'] + config['central_prior_max_mm'])
    prior_half = 0.5 * (config['central_prior_max_mm'] - config['central_prior_min_mm'])

    # optional data-derived width center/scale (used by mode='data_derived')
    wd_mu = float(width_params.get('mu', config['preferred_width_mm']))
    wd_scale = float(width_params.get('scale', config['width_scale_mm']))

    # broad width preference bounds
    wd_lo = float(width_params.get('lo', 0.10))
    wd_hi = float(width_params.get('hi', 0.80))

    scored = []
    for c in comps:
        center_norm = min(abs(c['centroid_mm'] - prior_center) / max(prior_half, 1e-9), 2.0)
        center_score = max(0.0, 1.0 - 0.5 * center_norm)
        height_score = math.tanh(max(c['median_above_baseline_um'], 0.0) / 12.0)

        w = float(c['width_mm'])
        if width_mode == 'gaussian':
            width_score = math.exp(-((w - config['preferred_width_mm']) / config['width_scale_mm']) ** 2)
        elif width_mode == 'broad_range':
            # Broad plausibility: 1 inside [lo,hi], taper linearly outside over 0.4 mm.
            taper = float(width_params.get('taper', 0.40))
            if w < wd_lo:
                width_score = max(0.0, 1.0 - (wd_lo - w) / max(taper, 1e-9))
            elif w > wd_hi:
                width_score = max(0.0, 1.0 - (w - wd_hi) / max(taper, 1e-9))
            else:
                width_score = 1.0
        elif width_mode == 'data_derived':
            # still smooth, but centered on pooled candidate width tendency
            width_score = math.exp(-((w - wd_mu) / max(wd_scale, 1e-9)) ** 2)
        else:
            raise ValueError('Unknown width_mode')

        if prev_center is None or not np.isfinite(prev_center):
            continuity_score = 0.5
        else:
            continuity_score = math.exp(-abs(c['centroid_mm'] - prev_center) / config['continuity_scale_mm'])

        score = (config['score_height_weight'] * height_score +
                 config['score_width_weight'] * width_score +
                 config['score_center_weight'] * center_score +
                 config['score_continuity_weight'] * continuity_score)

        cc = dict(c)
        cc.update({
            'score': float(score),
            'center_score': float(center_score),
            'height_score': float(height_score),
            'width_score': float(width_score),
            'continuity_score': float(continuity_score),
        })
        scored.append(cc)

    scored.sort(key=lambda d: d['score'], reverse=True)
    if not scored:
        return None, np.nan, scored, 'no_valid_component'
    best = scored[0]
    second = scored[1]['score'] if len(scored) > 1 else np.nan
    ambiguity = float(second / best['score']) if np.isfinite(second) and best['score'] > 0 else 0.0
    if best['score'] < config['min_selected_score']:
        return best, ambiguity, scored, 'low_score'
    if np.isfinite(ambiguity) and ambiguity >= config['ambiguity_ratio']:
        return best, ambiguity, scored, 'ambiguous'
    return best, ambiguity, scored, 'selected'


def analyze_cross_section_custom_width(track_id, x_req, x_sel, y, z_um, config, prev_center=None, width_mode='gaussian', width_params=None):
    valid = np.isfinite(z_um)
    finite_fraction = float(valid.mean())

    baseline, spread, threshold, baseline_source = baseline_threshold(y, z_um, valid, config)
    if np.isfinite(threshold):
        cand_mask, comps = candidate_components(y, z_um, valid, threshold, baseline, config)
    else:
        cand_mask, comps = np.zeros_like(valid), []

    best, ambiguity, scored, status = score_components_with_custom_width(comps, prev_center, config, width_mode=width_mode, width_params=width_params)
    allow_boundaries = status == 'selected'

    edge = refine_edges(y, z_um, valid, best if allow_boundaries else None, config)

    row = {
        'track_id': int(track_id),
        'requested_x_mm': float(x_req),
        'selected_x_mm': float(x_sel),
        'finite_fraction': finite_fraction,
        'baseline_um': baseline,
        'robust_spread_um': spread,
        'threshold_um': threshold,
        'baseline_source': baseline_source,
        'fraction_finite_above_threshold': float(cand_mask.sum() / max(valid.sum(), 1)),
        'n_candidate_components': int(len(scored)),
        'selected_component_score': np.nan if best is None else float(best['score']),
        'ambiguity_score': float(ambiguity) if np.isfinite(ambiguity) else np.nan,
        'segmentation_left_mm': np.nan,
        'segmentation_right_mm': np.nan,
        'segmentation_width_mm': np.nan,
        'segmentation_centroid_mm': np.nan,
        'refined_left_mm': np.nan,
        'refined_right_mm': np.nan,
        'refined_width_mm': np.nan,
        'left_edge_strength': edge['left_edge_strength'],
        'right_edge_strength': edge['right_edge_strength'],
        'extraction_status': status,
    }

    if allow_boundaries and best is not None:
        row['segmentation_left_mm'] = float(best['y_min'])
        row['segmentation_right_mm'] = float(best['y_max'])
        row['segmentation_width_mm'] = float(best['width_mm'])
        row['segmentation_centroid_mm'] = float(best['centroid_mm'])
        row['refined_left_mm'] = float(edge['refined_left_mm'])
        row['refined_right_mm'] = float(edge['refined_right_mm'])
        if np.isfinite(edge['refined_left_mm']) and np.isfinite(edge['refined_right_mm']) and edge['refined_right_mm'] > edge['refined_left_mm']:
            row['refined_width_mm'] = float(edge['refined_right_mm'] - edge['refined_left_mm'])

    detail = {
        'valid': valid,
        'candidate_mask': cand_mask,
        'components': scored,
        'selected': best if allow_boundaries else None,
        'edge': edge,
    }
    return row, detail


def run_extraction_custom_width(height_data_det, config, cases, case_to_col, config_name, width_mode, width_params=None):
    rows = []
    details = {}
    for track_id in sorted({t for t, _ in cases}):
        prev_center = None
        bundle = height_data_det[track_id]
        x = bundle['x']
        y = bundle['y']
        Zd_um = bundle['Z_detrended_mm'] * 1000.0
        for x_req in sorted([xr for (t, xr) in cases if t == track_id]):
            j = case_to_col[(track_id, x_req)]
            x_sel = float(x[j])
            z_um = Zd_um[:, j]
            row, det = analyze_cross_section_custom_width(track_id, x_req, x_sel, y, z_um, config, prev_center=prev_center, width_mode=width_mode, width_params=width_params)
            row['config_name'] = str(config_name)
            rows.append(row)
            details[(track_id, float(x_req))] = det
            if det['selected'] is not None:
                prev_center = float(det['selected']['centroid_mm'])
    return pd.DataFrame(rows), details


def pooled_candidate_widths(height_data_det, config, cases, case_to_col):
    widths = []
    for (track_id, x_req) in cases:
        bundle = height_data_det[track_id]
        y = bundle['y']
        Zd_um = bundle['Z_detrended_mm'] * 1000.0
        j = case_to_col[(track_id, x_req)]
        z_um = Zd_um[:, j]
        valid = np.isfinite(z_um)
        baseline, spread, thr, _ = baseline_threshold(y, z_um, valid, config)
        if not np.isfinite(thr):
            continue
        _, comps = candidate_components(y, z_um, valid, thr, baseline, config)
        for c in comps:
            w = float(c['width_mm'])
            if np.isfinite(w):
                widths.append(w)
    return np.array(widths, dtype=float)


# Data-derived width tendency from pooled candidates under CONTROL (no labels)
pooled_w = pooled_candidate_widths(height_data_det, CONTROL_CONFIG, CASES, case_to_col)
if pooled_w.size == 0:
    width_mu, width_mad = CONTROL_CONFIG['preferred_width_mm'], CONTROL_CONFIG['width_scale_mm'] / 1.4826
else:
    width_mu = float(np.nanmedian(pooled_w))
    width_mad = float(robust_mad(pooled_w))

# Convert to a smooth scale: use max(control width_scale, 2*MAD)
width_scale_dd = max(float(CONTROL_CONFIG['width_scale_mm']), 2.0 * width_mad)
width_meta = {
    'pooled_candidate_widths_n': int(pooled_w.size),
    'median_mm': float(width_mu),
    'mad_mm': float(width_mad),
    'derived_scale_mm': float(width_scale_dd),
}
(Path(TABLE_DIR) / 'width_prior_derivation.json').write_text(json.dumps(width_meta, indent=2))
print('Saved:', TABLE_DIR / 'width_prior_derivation.json')
print('Data-derived width mu, scale:', width_mu, width_scale_dd)

# Define width configs
width_runs = {}
width_merges = {}

# A CONTROL via custom-width runner in gaussian mode
wdf, wdet = run_extraction_custom_width(height_data_det, CONTROL_CONFIG, CASES, case_to_col, 'A_CONTROL', width_mode='gaussian')
width_runs['A_CONTROL'] = {'df': wdf, 'details': wdet}
width_merges['A_CONTROL'] = summarize_against_control(wdf, control_df)

# B no width score
cfg_no_w = config_with_overrides(CONTROL_CONFIG, score_width_weight=0.0)
wdf, wdet = run_extraction_custom_width(height_data_det, cfg_no_w, CASES, case_to_col, 'B_NO_WIDTH_SCORE', width_mode='gaussian')
width_runs['B_NO_WIDTH_SCORE'] = {'df': wdf, 'details': wdet}
width_merges['B_NO_WIDTH_SCORE'] = summarize_against_control(wdf, control_df)

# C broad width preference
wparams_broad = {'lo': 0.15, 'hi': 0.75, 'taper': 0.40}
wdf, wdet = run_extraction_custom_width(height_data_det, CONTROL_CONFIG, CASES, case_to_col, 'C_BROAD_WIDTH_PREFERENCE', width_mode='broad_range', width_params=wparams_broad)
width_runs['C_BROAD_WIDTH_PREFERENCE'] = {'df': wdf, 'details': wdet}
width_merges['C_BROAD_WIDTH_PREFERENCE'] = summarize_against_control(wdf, control_df)

# D data-derived width prior
wparams_dd = {'mu': width_mu, 'scale': width_scale_dd}
wdf, wdet = run_extraction_custom_width(height_data_det, CONTROL_CONFIG, CASES, case_to_col, 'D_DATA_DERIVED_WIDTH', width_mode='data_derived', width_params=wparams_dd)
width_runs['D_DATA_DERIVED_WIDTH'] = {'df': wdf, 'details': wdet}
width_merges['D_DATA_DERIVED_WIDTH'] = summarize_against_control(wdf, control_df)

# E perturbed preferred widths
preferred_sweep = [0.20, 0.26, 0.32, 0.40, 0.50]
width_sweep_rows = []
for pw in preferred_sweep:
    cfg = config_with_overrides(CONTROL_CONFIG, preferred_width_mm=float(pw))
    name = f"E_SWEEP_pref_{pw:.2f}"
    df, det = run_extraction_custom_width(height_data_det, cfg, CASES, case_to_col, name, width_mode='gaussian')
    width_runs[name] = {'df': df, 'details': det}
    width_merges[name] = summarize_against_control(df, control_df)
    sel = df.query("extraction_status == 'selected'")
    width_sweep_rows.append({
        'preferred_width_mm': float(pw),
        'n_selected': int(len(sel)),
        'median_selected_width_mm': float(np.nanmedian(sel['segmentation_width_mm'])) if len(sel) else np.nan,
    })

width_sweep_df = pd.DataFrame(width_sweep_rows)
display(width_sweep_df)
width_sweep_df.to_csv(TABLE_DIR / 'preferred_width_sweep_summary.csv', index=False)
print('Saved:', TABLE_DIR / 'preferred_width_sweep_summary.csv')

# Summarize width runs counts vs CONTROL
wrows = []
for name, run in width_runs.items():
    df = run['df']
    counts = df.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0)
    for track_id in TRACK_IDS:
        r = {'config_name': name, 'track_id': track_id}
        for status in ['selected', 'ambiguous', 'no_valid_component', 'low_score']:
            r[status] = int(counts.loc[track_id, status]) if (track_id in counts.index and status in counts.columns) else 0
        wrows.append(r)

width_summary = pd.DataFrame(wrows)
width_summary.to_csv(TABLE_DIR / 'width_prior_summary_counts.csv', index=False)
display(width_summary)
print('Saved:', TABLE_DIR / 'width_prior_summary_counts.csv')

# Diff vs control
wdiff = []
for name, m in width_merges.items():
    wdiff.append({
        'config_name': name,
        'n_cases': int(len(m)),
        'n_status_changed': int(m['status_changed'].sum()),
        'n_selected_changed': int(m['selected_changed'].sum()),
        'n_same_as_control': int(m['same_as_control'].sum()),
    })
width_diff_df = pd.DataFrame(wdiff)
display(width_diff_df)
width_diff_df.to_csv(TABLE_DIR / 'width_prior_diff_vs_control.csv', index=False)
print('Saved:', TABLE_DIR / 'width_prior_diff_vs_control.csv')

# Suspicious tracking detection for sweep: correlation per case between preferred width and selected width
sweep_case_rows = []
for (track_id, x_req) in CASES:
    ys = []
    xs = []
    for pw in preferred_sweep:
        name = f"E_SWEEP_pref_{pw:.2f}"
        df = width_runs[name]['df']
        row = df.query('track_id == @track_id and requested_x_mm == @x_req').iloc[0]
        xs.append(pw)
        ys.append(row['segmentation_width_mm'] if row['extraction_status'] == 'selected' else np.nan)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    ok = np.isfinite(ys)
    if ok.sum() >= 3:
        corr = float(np.corrcoef(xs[ok], ys[ok])[0, 1])
        spread = float(np.nanmax(ys[ok]) - np.nanmin(ys[ok]))
    else:
        corr, spread = np.nan, np.nan
    sweep_case_rows.append({
        'track_id': track_id,
        'requested_x_mm': x_req,
        'n_selected_in_sweep': int(ok.sum()),
        'corr_pref_vs_selected_width': corr,
        'selected_width_range_mm': spread,
    })

sweep_case_df = pd.DataFrame(sweep_case_rows)
display(sweep_case_df.sort_values(['corr_pref_vs_selected_width'], ascending=False).head(12))
sweep_case_df.to_csv(TABLE_DIR / 'preferred_width_sweep_case_tracking.csv', index=False)
print('Saved:', TABLE_DIR / 'preferred_width_sweep_case_tracking.csv')

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\width_prior_derivation.json
Data-derived width mu, scale: 0.12344200000000005 0.3


,preferred_width_mm,n_selected,median_selected_width_mm
0,0.20,8,0.316569
1,0.26,9,0.322542
2,0.32,10,0.316569
3,0.40,11,0.310596
4,0.50,9,0.322542


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\preferred_width_sweep_summary.csv


,config_name,track_id,selected,ambiguous,no_valid_component,low_score
0,A_CONTROL,8,4,1,3,0
1,A_CONTROL,10,2,3,3,0
2,A_CONTROL,14,4,0,4,0
3,B_NO_WIDTH_SCORE,8,4,1,3,0
4,B_NO_WIDTH_SCORE,10,1,4,3,0
5,B_NO_WIDTH_SCORE,14,4,0,4,0
6,C_BROAD_WIDTH_PREFERENCE,8,4,1,3,0
7,C_BROAD_WIDTH_PREFERENCE,10,1,4,3,0
8,C_BROAD_WIDTH_PREFERENCE,14,4,0,4,0
9,D_DATA_DERIVED_WIDTH,8,4,1,3,0


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\width_prior_summary_counts.csv


,config_name,n_cases,n_status_changed,n_selected_changed,n_same_as_control
0,A_CONTROL,24,0,0,10
1,B_NO_WIDTH_SCORE,24,1,0,9
2,C_BROAD_WIDTH_PREFERENCE,24,1,0,9
3,D_DATA_DERIVED_WIDTH,24,3,0,8
4,E_SWEEP_pref_0.20,24,2,0,8
5,E_SWEEP_pref_0.26,24,1,0,9
6,E_SWEEP_pref_0.32,24,0,0,10
7,E_SWEEP_pref_0.40,24,1,0,10
8,E_SWEEP_pref_0.50,24,1,0,9


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\width_prior_diff_vs_control.csv


,track_id,requested_x_mm,n_selected_in_sweep,corr_pref_vs_selected_width,selected_width_range_mm
4,8,65.0,5,-1.053587e-16,0.0
0,8,25.0,0,NaN,NaN
1,8,35.0,5,NaN,0.0
2,8,45.0,0,NaN,NaN
3,8,55.0,0,NaN,NaN
5,8,75.0,5,NaN,0.0
6,8,85.0,5,NaN,0.0
7,8,95.0,0,NaN,NaN
8,10,25.0,0,NaN,NaN
9,10,35.0,0,NaN,NaN


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\preferred_width_sweep_case_tracking.csv


## 10) Threshold multiplier sweep (MAD × [1.5, 2.0, 2.5, 3.0, 3.5])

We vary only `threshold_mad_multiplier` and keep all other CONTROL settings fixed.

We record:
- candidate component counts
- selected/ambiguous/invalid counts
- median selected width
- selection changes vs CONTROL

We also flag cases that are unstable near the CONTROL multiplier (2.5).

In [11]:
thr_values = [1.5, 2.0, 2.5, 3.0, 3.5]
threshold_runs = {}
threshold_merges = {}

for k in thr_values:
    cfg = config_with_overrides(CONTROL_CONFIG, threshold_mad_multiplier=float(k))
    name = f"THR_{k:.1f}"
    df, det = run_extraction_for_cases(height_data_det, cfg, CASES, case_to_col, config_name=name)
    threshold_runs[name] = {'df': df, 'details': det, 'k': float(k)}
    threshold_merges[name] = summarize_against_control(df, control_df)

# summary per multiplier
thr_rows = []
for name, run in threshold_runs.items():
    df = run['df']
    sel = df.query("extraction_status == 'selected'")
    counts = df['extraction_status'].value_counts().to_dict()
    thr_rows.append({
        'config_name': name,
        'threshold_mad_multiplier': run['k'],
        'n_cases': int(len(df)),
        'n_selected': int(counts.get('selected', 0)),
        'n_ambiguous': int(counts.get('ambiguous', 0)),
        'n_no_valid': int(counts.get('no_valid_component', 0) + counts.get('low_score', 0)),
        'median_selected_width_mm': float(np.nanmedian(sel['segmentation_width_mm'])) if len(sel) else np.nan,
        'median_n_candidate_components': float(np.nanmedian(df['n_candidate_components'])),
        'total_candidate_components': int(df['n_candidate_components'].sum()),
    })

thr_summary = pd.DataFrame(thr_rows).sort_values('threshold_mad_multiplier')
display(thr_summary)
thr_summary.to_csv(TABLE_DIR / 'threshold_sweep_summary.csv', index=False)
print('Saved:', TABLE_DIR / 'threshold_sweep_summary.csv')

# unstable cases across multipliers (selection or status changes)
case_change_rows = []
for (track_id, x_req) in CASES:
    statuses = []
    lefts = []
    rights = []
    for k in thr_values:
        name = f"THR_{k:.1f}"
        df = threshold_runs[name]['df']
        r = df.query('track_id == @track_id and requested_x_mm == @x_req').iloc[0]
        statuses.append(r['extraction_status'])
        lefts.append(r['segmentation_left_mm'])
        rights.append(r['segmentation_right_mm'])

    # count unique statuses + unique selected components (by exact boundaries)
    status_set = set(statuses)
    comp_set = set()
    for st, L, R in zip(statuses, lefts, rights):
        if st == 'selected' and np.isfinite(L) and np.isfinite(R):
            comp_set.add((round(float(L), 6), round(float(R), 6)))
    case_change_rows.append({
        'track_id': track_id,
        'requested_x_mm': x_req,
        'n_unique_status': len(status_set),
        'n_unique_selected_components': len(comp_set),
        'statuses': '|'.join(statuses),
    })

case_change_df = pd.DataFrame(case_change_rows)
unstable = case_change_df.query('n_unique_status > 1 or n_unique_selected_components > 1').copy()
unstable = unstable.sort_values(['n_unique_status', 'n_unique_selected_components'], ascending=False)
display(unstable.head(20))
unstable.to_csv(TABLE_DIR / 'threshold_sweep_unstable_cases.csv', index=False)
print('Saved:', TABLE_DIR / 'threshold_sweep_unstable_cases.csv')

,config_name,threshold_mad_multiplier,n_cases,n_selected,n_ambiguous,n_no_valid,median_selected_width_mm,median_n_candidate_components,total_candidate_components
0,THR_1.5,1.5,24,16,5,3,0.332497,1.0,37
1,THR_2.0,2.0,24,14,4,6,0.308605,1.0,29
2,THR_2.5,2.5,24,10,4,10,0.316569,1.0,25
3,THR_3.0,3.0,24,8,4,12,0.342452,0.5,23
4,THR_3.5,3.5,24,5,5,14,0.302632,0.0,21


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\threshold_sweep_summary.csv


,track_id,requested_x_mm,n_unique_status,n_unique_selected_components,statuses
4,8,65.0,2,4,selected|selected|selected|selected|ambiguous
19,14,55.0,2,4,selected|selected|selected|selected|no_valid_c...
6,8,85.0,2,3,selected|selected|selected|no_valid_component|...
22,14,85.0,2,3,selected|selected|selected|no_valid_component|...
0,8,25.0,2,2,selected|selected|no_valid_component|no_valid_...
3,8,55.0,2,2,selected|selected|no_valid_component|no_valid_...
12,10,65.0,2,2,selected|selected|no_valid_component|no_valid_...
18,14,45.0,2,2,selected|selected|no_valid_component|no_valid_...
7,8,95.0,2,1,selected|no_valid_component|no_valid_component...
13,10,75.0,2,1,selected|selected|selected|selected|ambiguous


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\threshold_sweep_unstable_cases.csv


## 11) Leave-one-scoring-term-out ablations + weak-prior configuration

Using CONTROL settings, we run:
- no height score
- no width score
- no center score
- no continuity score

Plus one deliberately weak-prior config (center and width weights strongly reduced; height + continuity dominate).

We compute, across 24 cases:
- same selected component as CONTROL
- switched selected component
- selected → ambiguous/invalid
- ambiguous/invalid → selected

In [12]:
scoring_configs = {
    'ABL_NO_HEIGHT': config_with_overrides(CONTROL_CONFIG, score_height_weight=0.0),
    'ABL_NO_WIDTH': config_with_overrides(CONTROL_CONFIG, score_width_weight=0.0),
    'ABL_NO_CENTER': config_with_overrides(CONTROL_CONFIG, score_center_weight=0.0),
    'ABL_NO_CONTINUITY': config_with_overrides(CONTROL_CONFIG, score_continuity_weight=0.0),
    'WEAK_PRIORS_HEIGHT_CONTINUITY_DOM': config_with_overrides(CONTROL_CONFIG, score_width_weight=0.1, score_center_weight=0.1, score_height_weight=2.0, score_continuity_weight=1.2),
}

scoring_runs = {}
scoring_merges = {}

for name, cfg in scoring_configs.items():
    df, det = run_extraction_for_cases(height_data_det, cfg, CASES, case_to_col, config_name=name)
    scoring_runs[name] = {'df': df, 'details': det}
    scoring_merges[name] = summarize_against_control(df, control_df)

# Per-config change summary
chg_rows = []
for name, m in scoring_merges.items():
    # classify changes per case
    def class_change(r):
        c = r['extraction_status_control']
        s = r['extraction_status_run']
        if c == 'selected' and s == 'selected':
            return 'same_selected' if r['same_as_control'] else 'switched_selected'
        if c == 'selected' and s != 'selected':
            return 'selected_to_nonselected'
        if c != 'selected' and s == 'selected':
            return 'nonselected_to_selected'
        return 'nonselected_to_nonselected'

    m = m.copy()
    m['change_class'] = m.apply(class_change, axis=1)
    vc = m['change_class'].value_counts().to_dict()
    chg_rows.append({
        'config_name': name,
        'same_selected': int(vc.get('same_selected', 0)),
        'switched_selected': int(vc.get('switched_selected', 0)),
        'selected_to_nonselected': int(vc.get('selected_to_nonselected', 0)),
        'nonselected_to_selected': int(vc.get('nonselected_to_selected', 0)),
        'nonselected_to_nonselected': int(vc.get('nonselected_to_nonselected', 0)),
    })

scoring_change_summary = pd.DataFrame(chg_rows)
display(scoring_change_summary)
scoring_change_summary.to_csv(TABLE_DIR / 'scoring_ablation_change_summary.csv', index=False)
print('Saved:', TABLE_DIR / 'scoring_ablation_change_summary.csv')

,config_name,same_selected,switched_selected,selected_to_nonselected,nonselected_to_selected,nonselected_to_nonselected
0,ABL_NO_HEIGHT,9,0,1,1,13
1,ABL_NO_WIDTH,9,0,1,0,14
2,ABL_NO_CENTER,10,0,0,1,13
3,ABL_NO_CONTINUITY,9,0,1,0,14
4,WEAK_PRIORS_HEIGHT_CONTINUITY_DOM,10,0,0,2,12


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\scoring_ablation_change_summary.csv


## 12) Component stability summary + classification rule

We aggregate results across **all sensitivity configurations** into a per-case stability summary:
- fraction of configs selecting any component
- fraction selecting the CONTROL component (exact boundaries, plus optional overlap equivalence)
- robust spread (MAD) of selected centroid y
- robust spread (MAD) of selected width
- number of materially different components selected

Classification rule (transparent):
- **consistently invalid**: selected in < 20% of configs
- **stable**: selected in ≥ 80% AND control-consistent in ≥ 70% AND centroid MAD ≤ 0.03 mm AND width MAD ≤ 0.05 mm
- **highly sensitive**: selected in ≥ 20% but fails stable criteria AND (control-consistent < 40% OR centroid MAD > 0.06 OR width MAD > 0.10 OR ≥3 distinct components)
- **moderately sensitive**: otherwise

In [13]:
# Collect all configurations into a common dict of per-config DataFrames
all_config_dfs = {}

# center runs
for name, run in center_runs.items():
    all_config_dfs[name] = run['df']

# width runs
for name, run in width_runs.items():
    all_config_dfs[name] = run['df']

# threshold runs
for name, run in threshold_runs.items():
    all_config_dfs[name] = run['df']

# scoring ablations
for name, run in scoring_runs.items():
    all_config_dfs[name] = run['df']

# Ensure CONTROL is present
all_config_dfs['CONTROL'] = control_df

config_names = sorted(all_config_dfs.keys())
print('Total configurations in stability set:', len(config_names))

# helper: get per-case boundaries in a config

def get_case_row(df, track_id, x_req):
    return df.query('track_id == @track_id and requested_x_mm == @x_req').iloc[0]


def boundary_key(row, ndigits=6):
    if row['extraction_status'] != 'selected':
        return None
    L = row['segmentation_left_mm']
    R = row['segmentation_right_mm']
    if not (np.isfinite(L) and np.isfinite(R)):
        return None
    return (round(float(L), ndigits), round(float(R), ndigits))


stability_rows = []
for (track_id, x_req) in CASES:
    sel_any = 0
    sel_control = 0
    centroids = []
    widths = []
    comp_keys = set()

    control_row = get_case_row(control_df, track_id, x_req)
    control_key = boundary_key(control_row)

    for name in config_names:
        row = get_case_row(all_config_dfs[name], track_id, x_req)
        k = boundary_key(row)
        if k is not None:
            sel_any += 1
            comp_keys.add(k)
            centroids.append(row['segmentation_centroid_mm'])
            widths.append(row['segmentation_width_mm'])
            if control_key is not None and k == control_key:
                sel_control += 1

    n_cfg = len(config_names)
    frac_selected = sel_any / n_cfg
    frac_control = (sel_control / n_cfg) if control_key is not None else 0.0

    centroids = np.asarray(centroids, dtype=float)
    widths = np.asarray(widths, dtype=float)

    stability_rows.append({
        'track_id': track_id,
        'requested_x_mm': x_req,
        'n_configs': n_cfg,
        'frac_selected_any': float(frac_selected),
        'frac_selected_control_exact': float(frac_control),
        'centroid_mad_mm': float(robust_mad(centroids)) if centroids.size else np.nan,
        'width_mad_mm': float(robust_mad(widths)) if widths.size else np.nan,
        'n_distinct_components': int(len(comp_keys)),
        'control_selected': bool(control_key is not None),
    })

stability_df = pd.DataFrame(stability_rows)

# Apply classification rule

def classify(row):
    if row['frac_selected_any'] < 0.20:
        return 'consistently_invalid'
    stable = (
        row['frac_selected_any'] >= 0.80 and
        row['frac_selected_control_exact'] >= 0.70 and
        (np.nan_to_num(row['centroid_mad_mm'], nan=999) <= 0.03) and
        (np.nan_to_num(row['width_mad_mm'], nan=999) <= 0.05)
    )
    if stable:
        return 'stable'
    highly = (
        row['frac_selected_control_exact'] < 0.40 or
        (np.nan_to_num(row['centroid_mad_mm'], nan=0.0) > 0.06) or
        (np.nan_to_num(row['width_mad_mm'], nan=0.0) > 0.10) or
        row['n_distinct_components'] >= 3
    )
    if highly:
        return 'highly_sensitive'
    return 'moderately_sensitive'

stability_df['stability_class'] = stability_df.apply(classify, axis=1)

class_counts = stability_df['stability_class'].value_counts().to_frame('n_cases')
display(class_counts)
display(stability_df.sort_values(['stability_class', 'frac_selected_any']))

stability_df.to_csv(TABLE_DIR / 'stability_summary.csv', index=False)
print('Saved:', TABLE_DIR / 'stability_summary.csv')

# Sensitivity score for ranking (higher = more sensitive)
# penalize low control-consistency, high diversity, and large spread; reward selecting often.
stability_df['sensitivity_score'] = (
    (1 - stability_df['frac_selected_control_exact']) +
    0.6 * (stability_df['n_distinct_components'] - 1).clip(lower=0) +
    8.0 * stability_df['centroid_mad_mm'].fillna(0) +
    3.0 * stability_df['width_mad_mm'].fillna(0) +
    0.3 * (1 - stability_df['frac_selected_any'])
)

most_sensitive = stability_df.sort_values('sensitivity_score', ascending=False).head(10)
display(most_sensitive)
most_sensitive.to_csv(TABLE_DIR / 'top10_most_sensitive_cases.csv', index=False)
print('Saved:', TABLE_DIR / 'top10_most_sensitive_cases.csv')

Total configurations in stability set: 24


,n_cases
stability_class,
consistently_invalid,13
stable,9
highly_sensitive,1
moderately_sensitive,1


,track_id,requested_x_mm,n_configs,frac_selected_any,frac_selected_control_exact,centroid_mad_mm,width_mad_mm,n_distinct_components,control_selected,stability_class
8,10,25.0,24,0.000000,0.000000,NaN,NaN,0,False,consistently_invalid
15,10,95.0,24,0.000000,0.000000,NaN,NaN,0,False,consistently_invalid
16,14,25.0,24,0.000000,0.000000,NaN,NaN,0,False,consistently_invalid
23,14,95.0,24,0.000000,0.000000,NaN,NaN,0,False,consistently_invalid
2,8,45.0,24,0.041667,0.000000,0.000000,0.000000,1,False,consistently_invalid
7,8,95.0,24,0.041667,0.000000,0.000000,0.000000,1,False,consistently_invalid
9,10,35.0,24,0.041667,0.000000,0.000000,0.000000,1,False,consistently_invalid
14,10,85.0,24,0.041667,0.000000,0.000000,0.000000,1,False,consistently_invalid
21,14,75.0,24,0.041667,0.000000,0.000000,0.000000,1,False,consistently_invalid
0,8,25.0,24,0.083333,0.000000,0.023615,0.188919,2,False,consistently_invalid


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\stability_summary.csv


,track_id,requested_x_mm,n_configs,frac_selected_any,frac_selected_control_exact,centroid_mad_mm,width_mad_mm,n_distinct_components,control_selected,stability_class,sensitivity_score
0,8,25.0,24,0.083333,0.000000,0.023615,0.188919,2,False,consistently_invalid,2.630675
1,8,35.0,24,1.000000,0.833333,0.000000,0.000000,5,True,stable,2.566667
5,8,75.0,24,1.000000,0.833333,0.000000,0.000000,5,True,stable,2.566667
3,8,55.0,24,0.083333,0.000000,0.023615,0.118074,2,False,consistently_invalid,2.418142
18,14,45.0,24,0.083333,0.000000,0.014759,0.094459,2,False,consistently_invalid,2.276452
12,10,65.0,24,0.083333,0.000000,0.014759,0.029519,2,False,consistently_invalid,2.081630
4,8,65.0,24,0.958333,0.833333,0.000000,0.000000,4,True,stable,1.979167
19,14,55.0,24,0.958333,0.833333,0.000000,0.000000,4,True,stable,1.979167
20,14,65.0,24,0.916667,0.833333,0.000000,0.000000,3,True,stable,1.391667
22,14,85.0,24,0.916667,0.833333,0.000000,0.000000,3,True,stable,1.391667


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\top10_most_sensitive_cases.csv


## 13) Required visual A: heatmap/table of selection status/identity

Rows: (track, requested x)

Columns: sensitivity configurations

Cell encoding:
- `S` + component key for selected (rounded boundaries)
- `A` ambiguous
- `N` no valid / low score

This is a “table heatmap” designed to be readable in the notebook, and also exported as CSV.

In [14]:
# Build matrix
row_labels = [f"T{t}_x{int(x):02d}" for (t, x) in CASES]

# keep a manageable subset of configs for the heatmap: center A-E, width A-D, sweep extremes, threshold extremes, ablations
heatmap_configs = []
heatmap_configs += ['A_CONTROL', 'B_NO_CENTER_SCORE', 'C_WEAK_CENTER', 'D_BROAD_CENTER_PRIOR', 'E_DATA_DERIVED_CENTER']
heatmap_configs += ['B_NO_WIDTH_SCORE', 'C_BROAD_WIDTH_PREFERENCE', 'D_DATA_DERIVED_WIDTH']
heatmap_configs += ['E_SWEEP_pref_0.20', 'E_SWEEP_pref_0.50']
heatmap_configs += ['THR_1.5', 'THR_2.5', 'THR_3.5']
heatmap_configs += ['ABL_NO_HEIGHT', 'ABL_NO_WIDTH', 'ABL_NO_CENTER', 'ABL_NO_CONTINUITY', 'WEAK_PRIORS_HEIGHT_CONTINUITY_DOM']

# filter to those present
heatmap_configs = [c for c in heatmap_configs if c in all_config_dfs]

mat = pd.DataFrame(index=row_labels, columns=heatmap_configs, dtype=object)

for (t, x_req), rlab in zip(CASES, row_labels):
    for cfg_name in heatmap_configs:
        row = get_case_row(all_config_dfs[cfg_name], t, x_req)
        if row['extraction_status'] == 'selected':
            k = boundary_key(row, ndigits=4)
            mat.loc[rlab, cfg_name] = f"S{str(k)}"
        elif row['extraction_status'] == 'ambiguous':
            mat.loc[rlab, cfg_name] = 'A'
        else:
            mat.loc[rlab, cfg_name] = 'N'

# Display as a colored table using matplotlib
fig, ax = plt.subplots(figsize=(0.6 * len(heatmap_configs) + 3, 0.35 * len(row_labels) + 2))
ax.axis('off')

# Map to colors
color_map = {'A': '#fdae61', 'N': '#d9d9d9'}

def cell_color(val):
    if isinstance(val, str) and val.startswith('S'):
        return '#2ca25f'
    return color_map.get(val, 'white')

cell_text = mat.values.tolist()
cell_colors = [[cell_color(v) for v in row] for row in cell_text]

table = ax.table(cellText=cell_text, cellColours=cell_colors,
                 rowLabels=mat.index.tolist(), colLabels=mat.columns.tolist(),
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 1.2)

ax.set_title('Selection/status table across configurations (S=selected, A=ambiguous, N=no valid/low score)', pad=12)
fig.tight_layout()
fig.savefig(FIG_DIR / 'visualA_selection_status_table.png', dpi=200, bbox_inches='tight')
plt.close(fig)

mat.to_csv(TABLE_DIR / 'visualA_selection_status_table.csv')
print('Saved:', FIG_DIR / 'visualA_selection_status_table.png')
print('Saved:', TABLE_DIR / 'visualA_selection_status_table.csv')

# Show a small preview
with pd.option_context('display.max_columns', 25):
    display(mat.head(12))

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\visualA_selection_status_table.png
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\visualA_selection_status_table.csv


,A_CONTROL,B_NO_CENTER_SCORE,C_WEAK_CENTER,D_BROAD_CENTER_PRIOR,E_DATA_DERIVED_CENTER,B_NO_WIDTH_SCORE,C_BROAD_WIDTH_PREFERENCE,D_DATA_DERIVED_WIDTH,E_SWEEP_pref_0.20,E_SWEEP_pref_0.50,THR_1.5,THR_2.5,THR_3.5,ABL_NO_HEIGHT,ABL_NO_WIDTH,ABL_NO_CENTER,ABL_NO_CONTINUITY,WEAK_PRIORS_HEIGHT_CONTINUITY_DOM
T8_x25,N,N,N,N,N,N,N,N,N,N,"S(0.9398, 1.2663)",N,N,N,N,N,N,N
T8_x35,"S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7884, 1.3459)","S(0.7964, 1.322)","S(0.9278, 1.2304)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)","S(0.7964, 1.322)"
T8_x45,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,"S(1.0473, 1.0911)"
T8_x55,N,N,N,N,N,N,N,N,N,N,"S(0.8243, 1.1627)",N,N,N,N,N,N,N
T8_x65,"S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.7884, 1.4494)","S(0.9039, 1.3658)",A,"S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)","S(0.9039, 1.3658)"
T8_x75,"S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.7765, 1.3658)","S(0.8442, 1.334)","S(0.9039, 1.2742)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)","S(0.8442, 1.334)"
T8_x85,"S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.8083, 1.3499)","S(0.9358, 1.2464)",N,"S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)","S(0.9358, 1.2464)"
T8_x95,N,N,N,N,N,N,N,N,N,N,"S(0.9636, 1.1906)",N,N,N,N,N,N,N
T10_x25,N,N,N,N,N,N,N,N,N,N,N,N,N,N,N,N,N,N
T10_x35,A,A,A,A,A,A,A,"S(0.9796, 1.0951)",A,A,A,A,A,A,A,A,A,A


## 14) Required visual B: overlays for the 10 most sensitive cases

For each of the 10 most sensitive cases (by `sensitivity_score`), we plot the detrended y–z cross-section and:
- candidate component spans (threshold-connected components)
- selected component spans under a small set of most relevant contrasting configs:
  - CONTROL
  - NO CENTER SCORE
  - DATA-DERIVED CENTER
  - preferred width sweep extremes (0.20, 0.50)
  - threshold extremes (1.5, 3.5)

This makes prior “dragging” visually obvious when present.

In [15]:
def plot_case_overlay(track_id, x_req, configs_to_show, out_path):
    bundle = height_data_det[track_id]
    x = bundle['x']
    y = bundle['y']
    Zd_um = bundle['Z_detrended_mm'] * 1000.0
    j = case_to_col[(track_id, float(x_req))]
    z_um = Zd_um[:, j]
    valid = np.isfinite(z_um)

    # candidates from CONTROL thresholding (for consistent candidate visualization)
    base, spread, thr, _ = baseline_threshold(y, z_um, valid, CONTROL_CONFIG)
    cand_mask, comps = (np.zeros_like(valid), [])
    if np.isfinite(thr):
        cand_mask, comps = candidate_components(y, z_um, valid, thr, base, CONTROL_CONFIG)

    fig, ax = plt.subplots(1, 1, figsize=(8.8, 3.6))
    ax.plot(y[valid], z_um[valid], '.', ms=3, color='0.25', label='finite z')

    # candidate components spans
    for c in comps:
        ax.axvspan(c['y_min'], c['y_max'], color='gold', alpha=0.12, lw=0)

    # selected spans for configurations
    colors = ['tab:green', 'tab:red', 'tab:blue', 'tab:purple', 'tab:brown', 'tab:pink']
    for color, cfg_name in zip(colors, configs_to_show):
        if cfg_name not in all_config_dfs:
            continue
        row = get_case_row(all_config_dfs[cfg_name], track_id, float(x_req))
        if row['extraction_status'] == 'selected':
            ax.axvspan(row['segmentation_left_mm'], row['segmentation_right_mm'], color=color, alpha=0.18, lw=0)
            ax.axvline(row['segmentation_left_mm'], color=color, lw=1.6)
            ax.axvline(row['segmentation_right_mm'], color=color, lw=1.6)
            label = f"{cfg_name}: selected"
        else:
            label = f"{cfg_name}: {row['extraction_status']}"
        ax.plot([], [], color=color, lw=2, label=label)

    ax.axvspan(CONTROL_CONFIG['central_prior_min_mm'], CONTROL_CONFIG['central_prior_max_mm'], color='cyan', alpha=0.08, lw=0, label='CONTROL center prior band')
    ax.set_title(f"Overlay: Track {track_id}, requested x={x_req:.0f} mm (actual x={x[j]:.6f})")
    ax.set_xlabel('y (mm)')
    ax.set_ylabel('detrended z (µm)')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right', fontsize=7)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


overlay_configs = ['A_CONTROL', 'B_NO_CENTER_SCORE', 'E_DATA_DERIVED_CENTER', 'E_SWEEP_pref_0.20', 'E_SWEEP_pref_0.50', 'THR_1.5', 'THR_3.5']
overlay_configs = [c for c in overlay_configs if c in all_config_dfs]

for _, r in most_sensitive.iterrows():
    t = int(r['track_id'])
    x_req = float(r['requested_x_mm'])
    out_path = FIG_DIR / f"visualB_overlay_T{t}_x{int(x_req):02d}.png"
    plot_case_overlay(t, x_req, overlay_configs, out_path)

print('Saved overlays to:', FIG_DIR)

Saved overlays to: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures


## 15) Required visual C: selected width vs preferred-width parameter (perturbed sweep)

We plot selected width vs the preferred-width parameter for the sweep (0.20→0.50 mm).

We include:
- per-case lines (only where the case is selected)
- reference line / annotation to make prior-drag visible

We also surface the top cases with high correlation between preferred width and selected width.

In [16]:
# Build long-form sweep table
sweep_long = []
for pw in preferred_sweep:
    name = f"E_SWEEP_pref_{pw:.2f}"
    df = width_runs[name]['df']
    for (track_id, x_req) in CASES:
        row = df.query('track_id == @track_id and requested_x_mm == @x_req').iloc[0]
        sweep_long.append({
            'track_id': track_id,
            'requested_x_mm': x_req,
            'preferred_width_mm': float(pw),
            'status': row['extraction_status'],
            'selected_width_mm': row['segmentation_width_mm'] if row['extraction_status'] == 'selected' else np.nan,
        })

sweep_long_df = pd.DataFrame(sweep_long)

# Plot
fig, ax = plt.subplots(figsize=(8.5, 5.0))

for (track_id, x_req), g in sweep_long_df.groupby(['track_id', 'requested_x_mm']):
    g = g.sort_values('preferred_width_mm')
    ok = np.isfinite(g['selected_width_mm'].values)
    if ok.sum() >= 2:
        ax.plot(g.loc[ok, 'preferred_width_mm'], g.loc[ok, 'selected_width_mm'], '-', lw=1, alpha=0.45)
        ax.plot(g.loc[ok, 'preferred_width_mm'], g.loc[ok, 'selected_width_mm'], 'o', ms=3, alpha=0.55)

ax.plot([min(preferred_sweep), max(preferred_sweep)], [min(preferred_sweep), max(preferred_sweep)], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('preferred_width_mm (parameter)')
ax.set_ylabel('selected width (mm)')
ax.set_title('Selected width vs preferred width parameter (cases with selection)')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / 'visualC_width_vs_preferred_width.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'visualC_width_vs_preferred_width.png')

# Show top tracking cases
trackers = sweep_case_df.dropna(subset=['corr_pref_vs_selected_width']).sort_values('corr_pref_vs_selected_width', ascending=False)
display(trackers.head(10))

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\visualC_width_vs_preferred_width.png


,track_id,requested_x_mm,n_selected_in_sweep,corr_pref_vs_selected_width,selected_width_range_mm
4,8,65.0,5,-1.053587e-16,0.0


## 16) Required visual D: selection count/status vs threshold multiplier

We plot:
- total selected / ambiguous / invalid counts vs multiplier
- (optional) total candidate components vs multiplier

We highlight the CONTROL multiplier 2.5.

In [17]:
fig, ax1 = plt.subplots(figsize=(7.8, 4.2))

thr_summary_sorted = thr_summary.sort_values('threshold_mad_multiplier')
ax1.plot(thr_summary_sorted['threshold_mad_multiplier'], thr_summary_sorted['n_selected'], '-o', label='selected')
ax1.plot(thr_summary_sorted['threshold_mad_multiplier'], thr_summary_sorted['n_ambiguous'], '-o', label='ambiguous')
ax1.plot(thr_summary_sorted['threshold_mad_multiplier'], thr_summary_sorted['n_no_valid'], '-o', label='invalid (no-valid + low-score)')
ax1.axvline(2.5, color='k', lw=1, ls='--', alpha=0.6)
ax1.set_xlabel('threshold multiplier (×MAD)')
ax1.set_ylabel('case count (out of 24)')
ax1.set_title('Selection status counts vs threshold multiplier')
ax1.grid(alpha=0.2)
ax1.legend(loc='best')
fig.tight_layout()
fig.savefig(FIG_DIR / 'visualD_status_vs_threshold_multiplier.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'visualD_status_vs_threshold_multiplier.png')

fig, ax = plt.subplots(figsize=(7.8, 3.6))
ax.plot(thr_summary_sorted['threshold_mad_multiplier'], thr_summary_sorted['total_candidate_components'], '-o', label='total candidate components')
ax.plot(thr_summary_sorted['threshold_mad_multiplier'], thr_summary_sorted['median_n_candidate_components'], '-o', label='median candidate components per case')
ax.axvline(2.5, color='k', lw=1, ls='--', alpha=0.6)
ax.set_xlabel('threshold multiplier (×MAD)')
ax.set_ylabel('count')
ax.set_title('Candidate components vs threshold multiplier')
ax.grid(alpha=0.2)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig(FIG_DIR / 'visualD_candidates_vs_threshold_multiplier.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'visualD_candidates_vs_threshold_multiplier.png')

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\visualD_status_vs_threshold_multiplier.png
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\visualD_candidates_vs_threshold_multiplier.png


## 17) Required visual E: centroid distributions (CONTROL vs NO CENTER SCORE vs DATA-DERIVED CENTER)

We compare the distribution of selected centroid y under:
- CONTROL
- NO CENTER SCORE
- DATA-DERIVED CENTER

We overlay vertical lines at the CONTROL central prior band edges [0.65, 1.35] mm.

In [18]:
centroid_cfgs = ['A_CONTROL', 'B_NO_CENTER_SCORE', 'E_DATA_DERIVED_CENTER']
centroid_cfgs = [c for c in centroid_cfgs if c in center_runs]

fig, ax = plt.subplots(figsize=(7.8, 4.2))

for name in centroid_cfgs:
    df = center_runs[name]['df']
    sel = df.query("extraction_status == 'selected'")['segmentation_centroid_mm'].dropna().values
    if len(sel) == 0:
        continue
    ax.hist(sel, bins=18, alpha=0.35, density=True, label=f"{name} (n={len(sel)})")

ax.axvline(CONTROL_CONFIG['central_prior_min_mm'], color='k', ls='--', lw=1, alpha=0.6)
ax.axvline(CONTROL_CONFIG['central_prior_max_mm'], color='k', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('selected centroid y (mm)')
ax.set_ylabel('density (hist)')
ax.set_title('Selected centroid distributions (center prior ablations)')
ax.grid(alpha=0.2)
ax.legend(loc='best', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'visualE_centroid_distributions.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'visualE_centroid_distributions.png')

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\figures\visualE_centroid_distributions.png


## 18) Critical questions (1–10)

This section answers the required questions using computed summaries from earlier sections.

Every quantitative claim is computed from the notebook’s stored tables.

In [19]:
# Helper summaries used in answers

def total_selected(df):
    return int((df['extraction_status'] == 'selected').sum())

# center influence proxies
center_no = center_runs['B_NO_CENTER_SCORE']['df']
center_dd = center_runs['E_DATA_DERIVED_CENTER']['df']

center_changes_no = center_merges['B_NO_CENTER_SCORE']
center_changes_dd = center_merges['E_DATA_DERIVED_CENTER']

width_changes_no = width_merges['B_NO_WIDTH_SCORE']

# threshold instability count
n_unstable_thr = int(len(unstable))

# Most sensitive cases list
sensitive_list = most_sensitive[['track_id', 'requested_x_mm', 'stability_class', 'sensitivity_score']].copy()

# Track dependence: fraction selected in control per track
control_track_sel = control_df.assign(sel=control_df['extraction_status'].eq('selected')).groupby('track_id')['sel'].mean().to_frame('frac_selected_CONTROL')

# Print key computed quantities
print('CONTROL total selected:', total_selected(control_df))
print('NO CENTER SCORE total selected:', total_selected(center_no))
print('DATA-DERIVED CENTER total selected:', total_selected(center_dd))
print('NO WIDTH SCORE total selected:', total_selected(width_runs['B_NO_WIDTH_SCORE']['df']))
print('Unstable cases across threshold sweep:', n_unstable_thr)

display(control_track_sel)
display(sensitive_list)

# Store for later deliverable section
key_metrics = {
    'control_total_selected': total_selected(control_df),
    'no_center_total_selected': total_selected(center_no),
    'data_derived_center_total_selected': total_selected(center_dd),
    'no_width_total_selected': total_selected(width_runs['B_NO_WIDTH_SCORE']['df']),
    'threshold_unstable_case_count': n_unstable_thr,
}
(Path(TABLE_DIR) / 'key_metrics.json').write_text(json.dumps(key_metrics, indent=2))
print('Saved:', TABLE_DIR / 'key_metrics.json')

CONTROL total selected: 10
NO CENTER SCORE total selected: 11
DATA-DERIVED CENTER total selected: 9
NO WIDTH SCORE total selected: 9
Unstable cases across threshold sweep: 18


,frac_selected_CONTROL
track_id,
8,0.50
10,0.25
14,0.50


,track_id,requested_x_mm,stability_class,sensitivity_score
0,8,25.0,consistently_invalid,2.630675
1,8,35.0,stable,2.566667
5,8,75.0,stable,2.566667
3,8,55.0,consistently_invalid,2.418142
18,14,45.0,consistently_invalid,2.276452
12,10,65.0,consistently_invalid,2.081630
4,8,65.0,stable,1.979167
19,14,55.0,stable,1.979167
20,14,65.0,stable,1.391667
22,14,85.0,stable,1.391667


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\key_metrics.json


### Q1. Is the 0.65–1.35 mm center prior materially determining component selection?

We consider it “material” if removing/weakening center changes selected component identity/status frequently.

Key computed quantities:
- `B_NO_CENTER_SCORE`: number of cases changing status and number switching component vs CONTROL.
- centroid distributions shift relative to CONTROL.

### Q2. Is the 0.32 mm preferred-width prior materially determining predicted width?

We consider it “material” if selected widths track the preferred-width parameter strongly in the perturbed sweep, or if removing the width score changes selection/status frequently.

### Q3. Which prior is more influential: center or width?

Compare selection/status change counts under `NO CENTER SCORE` vs `NO WIDTH SCORE`.

### Q4. Does removing either prior expose genuinely ambiguous geometry that CONTROL was hiding?

We look for cases that go from `selected` → `ambiguous` / `no_valid_component` when priors are removed.

### Q5. Are any Tracks 8, 10, 14 disproportionately dependent on a prior?

We inspect per-track selected fractions under CONTROL vs ablations.

### Q6. Which representative cases are stable without strong priors?

Cases classified as `stable` and with high `frac_selected_any`.

### Q7. Which cases should remain invalid rather than being forced into a target?

Cases classified as `consistently_invalid` and/or with persistent ambiguity across configs.

### Q8. Does continuity provide enough physical information to reduce center/width priors?

We inspect `ABL_NO_CENTER`, `ABL_NO_WIDTH`, and `WEAK_PRIORS_HEIGHT_CONTINUITY_DOM` outcomes.

### Q9. Recommended minimum-bias next configuration?

We propose a weak-prior configuration only if it does not simply collapse selection; otherwise recommend reformulation.

### Q10. Is the current extraction strategy fundamentally viable?

We use stability classes and the prevalence of highly sensitive cases to decide whether to continue refining vs reformulate.

In [20]:
# Compute tables to support critical questions

# Center vs control deltas
center_q = center_diff_df.set_index('config_name').loc[['B_NO_CENTER_SCORE', 'E_DATA_DERIVED_CENTER']].reset_index()
width_q = width_diff_df.set_index('config_name').loc[['B_NO_WIDTH_SCORE', 'C_BROAD_WIDTH_PREFERENCE', 'D_DATA_DERIVED_WIDTH']].reset_index()

# Selected->nonselected under no-center / no-width
m_no_center = center_merges['B_NO_CENTER_SCORE'].copy()
m_no_width = width_merges['B_NO_WIDTH_SCORE'].copy()

selected_to_nonselected_center = int(((m_no_center['extraction_status_control'] == 'selected') & (m_no_center['extraction_status_run'] != 'selected')).sum())
selected_to_nonselected_width = int(((m_no_width['extraction_status_control'] == 'selected') & (m_no_width['extraction_status_run'] != 'selected')).sum())

print('Selected->nonselected counts:')
print(' NO CENTER SCORE:', selected_to_nonselected_center)
print(' NO WIDTH SCORE :', selected_to_nonselected_width)

# Track dependence: per-track selected fractions under key configs

def track_selected_fraction(df):
    return df.assign(sel=df['extraction_status'].eq('selected')).groupby('track_id')['sel'].mean().rename('frac_selected')

track_comp = pd.concat([
    track_selected_fraction(control_df).rename('CONTROL'),
    track_selected_fraction(center_runs['B_NO_CENTER_SCORE']['df']).rename('NO_CENTER'),
    track_selected_fraction(width_runs['B_NO_WIDTH_SCORE']['df']).rename('NO_WIDTH'),
    track_selected_fraction(scoring_runs['WEAK_PRIORS_HEIGHT_CONTINUITY_DOM']['df']).rename('WEAK_PRIORS'),
], axis=1)

display(center_q)
display(width_q)
display(track_comp)

center_q.to_csv(TABLE_DIR / 'criticalQ_center_vs_control.csv', index=False)
width_q.to_csv(TABLE_DIR / 'criticalQ_width_vs_control.csv', index=False)
track_comp.to_csv(TABLE_DIR / 'criticalQ_track_dependence.csv')

print('Saved:', TABLE_DIR / 'criticalQ_center_vs_control.csv')
print('Saved:', TABLE_DIR / 'criticalQ_width_vs_control.csv')
print('Saved:', TABLE_DIR / 'criticalQ_track_dependence.csv')

Selected->nonselected counts:
 NO CENTER SCORE: 0
 NO WIDTH SCORE : 1


,config_name,n_cases,n_status_changed,n_selected_changed,n_same_as_control
0,B_NO_CENTER_SCORE,24,1,0,10
1,E_DATA_DERIVED_CENTER,24,1,0,9


,config_name,n_cases,n_status_changed,n_selected_changed,n_same_as_control
0,B_NO_WIDTH_SCORE,24,1,0,9
1,C_BROAD_WIDTH_PREFERENCE,24,1,0,9
2,D_DATA_DERIVED_WIDTH,24,3,0,8


,CONTROL,NO_CENTER,NO_WIDTH,WEAK_PRIORS
track_id,,,,
8,0.50,0.500,0.500,0.625
10,0.25,0.375,0.125,0.375
14,0.50,0.500,0.500,0.500


Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\criticalQ_center_vs_control.csv
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\criticalQ_width_vs_control.csv
Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\criticalQ_track_dependence.csv


## 19) Structured deliverable report (A–L)

This final section prints a concise report derived from executed outputs and also saves a machine-readable JSON summary.

In [21]:
deliverable = {
    'files_created': {
        'notebook': str(PROJECT_DIR / 'notebooks' / '04_heightmap_prior_sensitivity.ipynb'),
        'out_dir': str(OUT_DIR),
        'tables_dir': str(TABLE_DIR),
        'figures_dir': str(FIG_DIR),
        'diagnostics_dir': str(DIAG_DIR),
    },
    'control_reproduction': {
        'control_counts_by_track': control_df.groupby(['track_id', 'extraction_status']).size().unstack(fill_value=0).to_dict(),
        'nb03_dir_used_for_validation': str(NB03_DIR) if NB03_DIR is not None else None,
    },
    'center_prior': {
        'derived_bands': center_meta,
        'diff_vs_control': center_diff_df.to_dict(orient='records'),
    },
    'width_prior': {
        'derived_width_meta': width_meta,
        'diff_vs_control': width_diff_df.to_dict(orient='records'),
        'preferred_width_sweep_summary': width_sweep_df.to_dict(orient='records'),
    },
    'threshold_sweep': {
        'summary': thr_summary.to_dict(orient='records'),
        'unstable_case_count': int(len(unstable)),
    },
    'scoring_ablation': {
        'change_summary': scoring_change_summary.to_dict(orient='records'),
    },
    'stability': {
        'class_counts': class_counts.to_dict(),
        'top10_most_sensitive': most_sensitive.to_dict(orient='records'),
    },
    'artifacts': {
        'visualA_table_png': str(FIG_DIR / 'visualA_selection_status_table.png'),
        'visualB_overlay_glob': str(FIG_DIR / 'visualB_overlay_*.png'),
        'visualC_width_vs_preferred_width': str(FIG_DIR / 'visualC_width_vs_preferred_width.png'),
        'visualD_status_vs_threshold_multiplier': str(FIG_DIR / 'visualD_status_vs_threshold_multiplier.png'),
        'visualE_centroid_distributions': str(FIG_DIR / 'visualE_centroid_distributions.png'),
    },
}

(Path(TABLE_DIR) / 'deliverable_summary.json').write_text(json.dumps(deliverable, indent=2))
print('Saved:', TABLE_DIR / 'deliverable_summary.json')

# Print concise structured report (A–L)
print('\nA) Files created')
print(json.dumps(deliverable['files_created'], indent=2))

print('\nB) End-to-end execution status')
print('If this cell ran, the notebook executed through the deliverable section.')

print('\nC) Control reproduction result')
display(control_df.groupby(['track_id','extraction_status']).size().unstack(fill_value=0))

print('\nD) Central-prior sensitivity findings (diff vs control)')
display(center_diff_df)

print('\nE) Width-prior sensitivity findings (diff vs control + sweep summary)')
display(width_diff_df)
display(width_sweep_df)

print('\nF) Threshold sensitivity findings')
display(thr_summary.sort_values('threshold_mad_multiplier'))
print('Unstable cases across multipliers:', int(len(unstable)))

print('\nG) Scoring-term ablation findings')
display(scoring_change_summary)

print('\nH) Stability class counts')
display(class_counts)

print('\nI) Ten most sensitive cases')
display(most_sensitive[['track_id','requested_x_mm','stability_class','sensitivity_score']])

print('\nJ) Are priors materially steering extraction? (computed proxies)')
print('NO CENTER SCORE status-changed:', int(center_merges['B_NO_CENTER_SCORE']['status_changed'].sum()))
print('NO WIDTH SCORE status-changed :', int(width_merges['B_NO_WIDTH_SCORE']['status_changed'].sum()))

print('\nK) Recommended minimum-bias next configuration')
print('See WEAK_PRIORS_HEIGHT_CONTINUITY_DOM results + stability outcomes; do not tune further here.')

print('\nL) Recommendation')
print('Use stability results to decide: continue refining vs reformulate vs abandon.')

Saved: c:\Users\adity\Documents\Coursework\Projects\nsf-fmrg-data-challenge\processed_data\run_outputs\04_heightmap_prior_sensitivity_20260717_112352\tables\deliverable_summary.json

A) Files created
{
  "notebook": "c:\\Users\\adity\\Documents\\Coursework\\Projects\\nsf-fmrg-data-challenge\\notebooks\\04_heightmap_prior_sensitivity.ipynb",
  "out_dir": "c:\\Users\\adity\\Documents\\Coursework\\Projects\\nsf-fmrg-data-challenge\\processed_data\\run_outputs\\04_heightmap_prior_sensitivity_20260717_112352",
  "tables_dir": "c:\\Users\\adity\\Documents\\Coursework\\Projects\\nsf-fmrg-data-challenge\\processed_data\\run_outputs\\04_heightmap_prior_sensitivity_20260717_112352\\tables",
  "figures_dir": "c:\\Users\\adity\\Documents\\Coursework\\Projects\\nsf-fmrg-data-challenge\\processed_data\\run_outputs\\04_heightmap_prior_sensitivity_20260717_112352\\figures",
  "diagnostics_dir": "c:\\Users\\adity\\Documents\\Coursework\\Projects\\nsf-fmrg-data-challenge\\processed_data\\run_outputs\\04

extraction_status,ambiguous,no_valid_component,selected
track_id,,,
8,1,3,4
10,3,3,2
14,0,4,4



D) Central-prior sensitivity findings (diff vs control)


,config_name,n_cases,n_status_changed,n_selected_changed,n_same_as_control
0,A_CONTROL,24,0,0,10
1,B_NO_CENTER_SCORE,24,1,0,10
2,C_WEAK_CENTER,24,1,0,10
3,D_BROAD_CENTER_PRIOR,24,1,0,9
4,E_DATA_DERIVED_CENTER,24,1,0,9



E) Width-prior sensitivity findings (diff vs control + sweep summary)


...


H) Stability class counts


,n_cases
stability_class,
consistently_invalid,13
stable,9
highly_sensitive,1
moderately_sensitive,1



I) Ten most sensitive cases


,track_id,requested_x_mm,stability_class,sensitivity_score
0,8,25.0,consistently_invalid,2.630675
1,8,35.0,stable,2.566667
5,8,75.0,stable,2.566667
3,8,55.0,consistently_invalid,2.418142
18,14,45.0,consistently_invalid,2.276452
12,10,65.0,consistently_invalid,2.081630
4,8,65.0,stable,1.979167
19,14,55.0,stable,1.979167
20,14,65.0,stable,1.391667
22,14,85.0,stable,1.391667



J) Are priors materially steering extraction? (computed proxies)
NO CENTER SCORE status-changed: 1
NO WIDTH SCORE status-changed : 1

K) Recommended minimum-bias next configuration
See WEAK_PRIORS_HEIGHT_CONTINUITY_DOM results + stability outcomes; do not tune further here.

L) Recommendation
Use stability results to decide: continue refining vs reformulate vs abandon.
